In [ ]:
# Cell 1 — Final TEST authorization and immutable contract
import hashlib, json, time
from pathlib import Path

AUTHORIZE_LOCKED_TEST = True
LOCKED_TEST_ACK = "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST"
PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]

EXPECTED_GATED_STEP = {42:600, 1337:200, 2026:600, 31415:400, 271828:932}
EXPECTED_FACETGATE_STEP = {42:600, 1337:932, 2026:400, 31415:600, 271828:600}

MODEL_ID = "google/gemma-4-12B-it"
MODEL_REVISION = "707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7"
MODEL_WEIGHTS_SHA256 = "5a84cb313260ac447237b890387116dfa8682e49a6b44bc585ae8353abbff18d"
FACETGATE_PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
COMMON_SCIENTIFIC_HASH = "0db67ec95431fa219e3f414ceaed1ec2c37774420a7b464246be2b69a3fa371c"
IMPLEMENTATION_CONTRACT_SHA256 = "224bdffa1b46db781d297ad8ae4732bdfce2f501c10cac022f9a8842c26b3c4b"
FACETGATE_BUILD_ID = "GEMMA4_FACETGATE_AUTOSEED_20260913_A"
LOCKED_TEST_BUILD_ID = "GEMMA4_FACETGATE_5SEED_LOCKED_TEST_ALL_PAPER_DATA_20260913_D_METADATAFIX_RESUME"

EXPECTED_LABEL_SHA256 = "9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6"
EXPECTED_OCR_SHA256 = "eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c"
EXPECTED_SPLIT_SHA256 = "bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0"
EXPECTED_CANONICAL_ROW_SHA256 = "440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39"
EXPECTED_TEST_IDENTITY_SHA256 = "5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db"
EXPECTED_IMAGE_BUNDLE_SHA256 = "9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af"

DEV_FEATURE_WORKER_SHA256 = "7dd04c94a4e91767767b826be217f306f280b30040a165152fd5c63b74041d85"
LOCKED_TEST_FEATURE_WORKER_SHA256 = "0d0da3269ea9e49bf6d996f915d5061fefbb4b6de5b5ed8a7807254cfb410ebe"

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK != "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final locked TEST is not explicitly authorized.")

print(json.dumps({
    "build_id": LOCKED_TEST_BUILD_ID,
    "architecture": "FacetGate",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "seeds": PUBLICATION_SEEDS,
    "expected_gated_steps": EXPECTED_GATED_STEP,
    "expected_facetgate_steps": EXPECTED_FACETGATE_STEP,
    "authorization": True,
    "locked_test_policy": "one frozen 932-row pass per seed; no TEST tuning",
}, indent=2))


In [ ]:
# Cell 2 — Pinned Kaggle runtime
# Protect the Kaggle base stack; install only the exact Gemma runtime used in development.
import importlib.metadata as md, json, subprocess, sys, os, time
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/M3Offend_9318_Gemma4_12B_FacetGate_5Seed_FINAL_LOCKED_TEST")
TEMP_ROOT = Path("/kaggle/temp/gemma4_facetgate_5seed_lock")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_START = time.time()

def dist_version(name):
    try: return md.version(name)
    except Exception: return None

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Select 2x Tesla T4.")
if torch.cuda.device_count()!=2:
    raise RuntimeError(f"Expected exactly 2 GPUs, got {torch.cuda.device_count()}")
gpu_names=[torch.cuda.get_device_name(i) for i in range(2)]
if not all("T4" in x.upper() for x in gpu_names):
    raise RuntimeError(f"Frozen Gemma4 execution requires 2x T4; got {gpu_names}")

protected=["torch","torchvision","Pillow","numpy","pandas","scipy","scikit-learn"]
before={k:dist_version(k) for k in protected}
if any(v is None for v in before.values()):
    raise RuntimeError(f"Missing protected package(s): {before}")
constraints=WORK_ROOT/"gemma4_locked_test_constraints.txt"
constraints.write_text("\n".join(f"{k}=={v}" for k,v in before.items())+"\n",encoding="utf-8")

required={"transformers":"5.15.0","peft":"0.20.0","accelerate":"1.14.0","bitsandbytes":"0.50.1","safetensors":"0.8.0"}
if any(dist_version(k)!=v for k,v in required.items()):
    cmd=[sys.executable,"-m","pip","install","--no-cache-dir","--upgrade-strategy","only-if-needed","-c",str(constraints)]
    cmd += [f"{k}=={v}" for k,v in required.items()]
    print("[PIP]"," ".join(cmd),flush=True)
    cp=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(cp.stdout,flush=True)
    if cp.returncode!=0:
        raise RuntimeError(f"Pinned Gemma runtime install failed rc={cp.returncode}")

after={k:dist_version(k) for k in protected}
if after!=before:
    raise RuntimeError(f"Protected Kaggle stack drifted. before={before} after={after}")
observed={k:dist_version(k) for k in required}
if observed!=required:
    raise RuntimeError(f"Pinned runtime mismatch expected={required} observed={observed}")

probe=r"""
import json
from PIL import Image
import torch, torchvision, transformers, peft, accelerate, bitsandbytes
from transformers import AutoProcessor, Gemma4UnifiedModel, BitsAndBytesConfig
im=Image.new("RGB",(2,2))
print(json.dumps({"torch":torch.__version__,"torchvision":torchvision.__version__,
"transformers":transformers.__version__,"peft":peft.__version__,
"accelerate":accelerate.__version__,"bitsandbytes":bitsandbytes.__version__,
"gemma4_import":True,"image_mode":im.mode},sort_keys=True))
"""
cp=subprocess.run([sys.executable,"-c",probe],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(cp.stdout,flush=True)
if cp.returncode!=0:
    raise RuntimeError("Fresh-interpreter Gemma4 import probe failed.")

print("[RUNTIME PASS]",{"gpus":gpu_names,"protected":after,"gemma_runtime":observed})


In [ ]:
# Cell 3 — Freeze all five FacetGate development packages and exact Gated parents
# CPU only. TEST remains unopened.
import os, json, hashlib, re, shutil
from pathlib import Path
import torch, numpy as np, pandas as pd

BINARY_TASKS=["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS=["Bangla","English","Banglish"]
TARGET_LABELS=["Individual","Gender","Political","Social","National","Religious","Others"]

def sha256_file(path,chunk=8<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def json_sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode()).hexdigest()

def atomic_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    os.replace(tmp,path)

def nearest_ancestor_file(path,name,max_up=6):
    p=Path(path)
    for _ in range(max_up+1):
        q=p/name
        if q.is_file(): return q
        if p.parent==p: break
        p=p.parent
    return None

def normalize_gated_thresholds(path):
    obj=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(obj,dict) and isinstance(obj.get("thresholds"),dict):
        t=obj["thresholds"]
    else:
        t=obj
    if not isinstance(t,dict):
        raise RuntimeError(f"Unexpected Gated threshold schema: {path}")
    binary={}
    target={}
    for task in BINARY_TASKS:
        if task not in t: raise RuntimeError(f"Missing binary threshold {task} in {path}")
        binary[task]=float(t[task])
    for lab in TARGET_LABELS:
        k=f"target::{lab}"
        if k not in t: raise RuntimeError(f"Missing target threshold {k} in {path}")
        target[lab]=float(t[k])
    return {"binary":binary,"target":target}

def normalize_facet_thresholds(path):
    obj=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(obj,dict) and isinstance(obj.get("thresholds"),dict) and "facetgate_target" not in obj:
        obj=obj["thresholds"]
    if not isinstance(obj,dict):
        raise RuntimeError(f"Unexpected FacetGate threshold schema: {path}")
    # Gemma development v1.0.1 writes these three dictionaries.
    if isinstance(obj.get("facetgate_target"),dict):
        facet={lab:float(obj["facetgate_target"][lab]) for lab in TARGET_LABELS}
    elif all(lab in obj for lab in TARGET_LABELS):
        facet={lab:float(obj[lab]) for lab in TARGET_LABELS}
    else:
        raise RuntimeError(f"Missing facetgate_target thresholds in {path}")
    gated=None
    if isinstance(obj.get("gated_target"),dict):
        gated={lab:float(obj["gated_target"][lab]) for lab in TARGET_LABELS}
    core=None
    if isinstance(obj.get("core5_binary"),dict):
        core={t:float(obj["core5_binary"][t]) for t in BINARY_TASKS}
    return {"facetgate_target":facet,"gated_target":gated,"core5_binary":core}

# ---- discover five completed Gemma FacetGate development roots ----
facet_candidates={s:[] for s in PUBLICATION_SEEDS}
for st in INPUT_ROOT.rglob("FACETGATE_DEVELOPMENT_COMPLETE.json"):
    try:
        obj=json.loads(st.read_text(encoding="utf-8"))
    except Exception:
        continue
    seed=int(obj.get("seed",-1))
    if seed not in facet_candidates: continue
    if obj.get("architecture")!="FacetGate" or obj.get("status")!="DEVELOPMENT_COMPLETE": continue
    root=st.parent
    art=root/"artifacts"
    req=[
        root/"run_config.json", art/"facetgate_result.json", art/"selected_facetgate_checkpoint.pt",
        art/"validation_thresholds.json", art/"facetgate_feature_cache.npz", art/"FEATURE_CACHE_MANIFEST.json"
    ]
    if not all(p.is_file() for p in req): continue
    try:
        rc=json.loads((root/"run_config.json").read_text(encoding="utf-8"))
        fr=json.loads((art/"facetgate_result.json").read_text(encoding="utf-8"))
        if rc.get("architecture")!="FacetGate" or rc.get("model_id")!=MODEL_ID or rc.get("model_revision")!=MODEL_REVISION: continue
        if rc.get("protocol_sha256")!=FACETGATE_PROTOCOL_SHA256: continue
        if rc.get("common_scientific_hash")!=COMMON_SCIENTIFIC_HASH: continue
        if rc.get("implementation_contract_sha256")!=IMPLEMENTATION_CONTRACT_SHA256: continue
        if int(rc.get("seed",-1))!=seed: continue
        if rc.get("execution_mode")!="REUSE_EXISTING_GATED": continue
        if int(obj.get("locked_test_model_forwards",-1))!=0 or int(fr.get("locked_test_model_forwards",-1))!=0: continue
        if int(obj.get("selected_gated_step",-1))!=EXPECTED_GATED_STEP[seed]: continue
        if int(obj.get("selected_facetgate_step",-1))!=EXPECTED_FACETGATE_STEP[seed]: continue
        parent=rc.get("parent") or {}
        if int(parent.get("manifest_payload",{}).get("seed",-1))!=seed: continue
        if int(parent.get("manifest_payload",{}).get("global_step",-1))!=EXPECTED_GATED_STEP[seed]: continue
        if parent.get("manifest_payload",{}).get("common_scientific_hash")!=COMMON_SCIENTIFIC_HASH: continue
        if parent.get("manifest_payload",{}).get("implementation_contract_sha256")!=IMPLEMENTATION_CONTRACT_SHA256: continue
        state=torch.load(art/"selected_facetgate_checkpoint.pt",map_location="cpu",weights_only=False)
        if int(state.get("seed",-1))!=seed or int(state.get("step",-1))!=EXPECTED_FACETGATE_STEP[seed]: continue
        if state.get("protocol_sha256")!=FACETGATE_PROTOCOL_SHA256: continue
        th=normalize_facet_thresholds(art/"validation_thresholds.json")
        key=(
            sha256_file(art/"selected_facetgate_checkpoint.pt"),
            sha256_file(art/"validation_thresholds.json"),
            sha256_file(art/"facetgate_result.json"),
            sha256_file(art/"facetgate_feature_cache.npz"),
            sha256_file(art/"FEATURE_CACHE_MANIFEST.json"),
        )
        facet_candidates[seed].append({
            "root":str(root),"status":obj,"run_config":rc,"facet_result":fr,
            "selected_facetgate_checkpoint":str(art/"selected_facetgate_checkpoint.pt"),
            "selected_facetgate_sha256":key[0],
            "validation_thresholds":str(art/"validation_thresholds.json"),
            "validation_threshold_sha256":key[1],
            "facetgate_thresholds":th,
            "feature_cache":str(art/"facetgate_feature_cache.npz"),
            "facet_result_sha256":key[2],
            "feature_cache_sha256":key[3],
            "feature_cache_manifest_sha256":key[4],
            "parent_checkpoint_content_sha256":parent.get("checkpoint_content_sha256"),
            "parent_record":parent,
            "dedupe_key":key,
        })
    except Exception:
        continue

FACET={}
facet_errors={}
for seed,cands in facet_candidates.items():
    uniq={}
    for x in cands: uniq.setdefault(x["dedupe_key"],x)
    if len(uniq)!=1:
        facet_errors[seed]={"candidate_count":len(cands),"unique_count":len(uniq),"roots":[x["root"] for x in cands]}
    else:
        FACET[seed]=next(iter(uniq.values()))
if facet_errors:
    raise RuntimeError("Could not uniquely freeze all five Gemma4 FacetGate development packages: "+json.dumps(facet_errors,indent=2))

# Deep immutable development-artifact audit.
# This is stronger than a numerical replay: the exact selected specialist checkpoint,
# validation thresholds, frozen feature cache, result JSON, and feature-cache manifest
# are all SHA-bound before TEST can be opened.
for seed in PUBLICATION_SEEDS:
    root=Path(FACET[seed]["root"])
    art=root/"artifacts"
    fcm_path=art/"FEATURE_CACHE_MANIFEST.json"
    result_path=art/"facetgate_result.json"
    if not fcm_path.is_file():
        raise RuntimeError(f"Seed {seed} FEATURE_CACHE_MANIFEST.json missing")
    fcm=json.loads(fcm_path.read_text(encoding="utf-8"))
    fr=FACET[seed]["facet_result"]
    observed_cache_sha=sha256_file(FACET[seed]["feature_cache"])
    expected_cache_sha=str(fcm.get("sha256",""))
    if observed_cache_sha!=expected_cache_sha:
        raise RuntimeError(
            f"Seed {seed} feature-cache SHA drift: observed={observed_cache_sha} expected={expected_cache_sha}"
        )
    if int(fcm.get("seed",-1))!=seed:
        raise RuntimeError(f"Seed {seed} feature-cache seed drift")
    if int(fcm.get("rows",-1))!=8386 or int(fcm.get("train",-1))!=7454 or int(fcm.get("validation",-1))!=932:
        raise RuntimeError(f"Seed {seed} feature-cache row contract drift: {fcm}")
    if int(fcm.get("TEST_forwards",-1))!=0:
        raise RuntimeError(f"Seed {seed} development feature cache touched TEST")
    if str(fr.get("verdict","")).upper()!="PASS":
        raise RuntimeError(f"Seed {seed} frozen development verdict is not PASS")
    if fr.get("execution_mode")!="REUSE_EXISTING_GATED":
        raise RuntimeError(f"Seed {seed} development execution_mode drift: {fr.get('execution_mode')}")
    if int(fr.get("selected_gated_step",-1))!=EXPECTED_GATED_STEP[seed]:
        raise RuntimeError(f"Seed {seed} selected Gated step drift in result JSON")
    if int(fr.get("selected_facetgate_step",-1))!=EXPECTED_FACETGATE_STEP[seed]:
        raise RuntimeError(f"Seed {seed} selected FacetGate step drift in result JSON")
    if int(fr.get("locked_test_model_forwards",-1))!=0:
        raise RuntimeError(f"Seed {seed} frozen development result touched TEST")
    # Threshold JSON must exactly match the thresholds serialized inside the frozen result.
    result_thr=(fr.get("thresholds") or {}).get("facetgate_target") or {}
    for lab in TARGET_LABELS:
        if lab not in result_thr:
            raise RuntimeError(f"Seed {seed} frozen result missing Target threshold for {lab}")
        if abs(float(result_thr[lab])-float(FACET[seed]["facetgate_thresholds"]["facetgate_target"][lab]))>1e-12:
            raise RuntimeError(f"Seed {seed} Target threshold drift for {lab}")
    FACET[seed]["feature_cache_sha256"]=observed_cache_sha
    FACET[seed]["feature_cache_manifest_sha256"]=sha256_file(fcm_path)
    FACET[seed]["facet_result_sha256"]=sha256_file(result_path)

# ---- discover exact same-seed Gated parents ----
gated_candidates={s:[] for s in PUBLICATION_SEEDS}
for mp in INPUT_ROOT.rglob("checkpoint_manifest.json"):
    try:
        m=json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    seed=int(m.get("seed",-1))
    if seed not in gated_candidates: continue
    if int(m.get("global_step",-1))!=EXPECTED_GATED_STEP[seed]: continue
    if m.get("common_scientific_hash")!=COMMON_SCIENTIFIC_HASH: continue
    if m.get("implementation_contract_sha256")!=IMPLEMENTATION_CONTRACT_SHA256: continue
    cp=mp.parent
    adapter=cp/"adapter"; heads=cp/"classifier_heads.pt"
    if not adapter.is_dir() or not heads.is_file(): continue
    if not (adapter/"adapter_config.json").is_file(): continue
    if not any((adapter/n).is_file() for n in ["adapter_model.safetensors","adapter_model.bin"]): continue
    run_cfg=nearest_ancestor_file(cp,"run_config.json",max_up=5)
    if run_cfg is None: continue
    try:
        rc=json.loads(run_cfg.read_text(encoding="utf-8"))
    except Exception:
        continue
    if rc.get("model_id")!=MODEL_ID or rc.get("model_revision")!=MODEL_REVISION: continue
    if int(rc.get("seed",-1))!=seed: continue
    if rc.get("common_scientific_hash")!=COMMON_SCIENTIFIC_HASH: continue
    if rc.get("implementation_contract_sha256")!=IMPLEMENTATION_CONTRACT_SHA256: continue
    root=run_cfg.parent
    thr=root/"artifacts"/"thresholds.json"
    if not thr.is_file(): continue
    try:
        gthr=normalize_gated_thresholds(thr)
    except Exception:
        continue
    expected_parent_sha=FACET[seed]["parent_checkpoint_content_sha256"]
    observed_parent_sha=m.get("checkpoint_content_sha256")
    if expected_parent_sha and observed_parent_sha!=expected_parent_sha: continue
    gated_candidates[seed].append({
        "path":str(cp),"manifest":str(mp),"manifest_payload":m,
        "adapter":str(adapter),"classifier_heads":str(heads),
        "thresholds":str(thr),"normalized_thresholds":gthr,
        "run_config":str(run_cfg),"gated_root":str(root),
        "checkpoint_content_sha256":observed_parent_sha,
        "dedupe_key":observed_parent_sha or sha256_file(heads),
        "historical_parent_test_artifacts_ignored":True,
    })

GATED={}
gated_errors={}
for seed,cands in gated_candidates.items():
    uniq={}
    for x in cands: uniq.setdefault(x["dedupe_key"],x)
    if len(uniq)!=1:
        gated_errors[seed]={"candidate_count":len(cands),"unique_count":len(uniq),"paths":[x["path"] for x in cands]}
    else:
        GATED[seed]=next(iter(uniq.values()))
if gated_errors:
    expected_sources={
        42:"sumaiyatubi / memeoffendbn-9318-gemma4-12b-gated-allinone-kaggle",
        1337:"mohammedmahatab / memeoffendbn-9318-gemma4-12b-gated-1337",
        2026:"asmahadistudent / memeoffendbn-9318-gemma4-12b-gated-2026",
        31415:"moriumbegum123 / memeoffendbn-9318-gemma4-12b-gated-31415",
        271828:"mrnone66 / memeoffendbn-9318-gemma4-12b-gated-271828",
    }
    raise RuntimeError(
        "Exact Gated parent weight outputs are required for final TEST feature extraction. "
        "The FacetGate development notebook outputs alone do not contain the parent LoRA adapter + classifier heads. "
        "Attach the five exact Gated notebook outputs. Missing/ambiguous="+json.dumps(gated_errors,indent=2)+
        "\nExpected historical sources="+json.dumps(expected_sources,indent=2)
    )

# Cross-bind development record to currently mounted Gated state.
for seed in PUBLICATION_SEEDS:
    if GATED[seed]["checkpoint_content_sha256"]!=FACET[seed]["parent_checkpoint_content_sha256"]:
        raise RuntimeError(f"Seed {seed} parent checkpoint SHA mismatch")
    # Ensure development package's parent selected step and current mounted checkpoint agree.
    if int(GATED[seed]["manifest_payload"]["global_step"])!=EXPECTED_GATED_STEP[seed]:
        raise RuntimeError(f"Seed {seed} Gated step drift")

freeze={
    "schema":"M3Offend_Gemma4_12B_FacetGate_5Seed_PRETEST_ASSET_FREEZE_v1_0",
    "build_id":LOCKED_TEST_BUILD_ID,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,
    "facetgate_protocol_sha256":FACETGATE_PROTOCOL_SHA256,
    "parent_gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "seeds":{},
    "TEST_model_forwards_before_open":0,
}
for seed in PUBLICATION_SEEDS:
    freeze["seeds"][str(seed)]={
        "gated_step":EXPECTED_GATED_STEP[seed],
        "gated_checkpoint_content_sha256":GATED[seed]["checkpoint_content_sha256"],
        "facetgate_step":EXPECTED_FACETGATE_STEP[seed],
        "facetgate_checkpoint_sha256":FACET[seed]["selected_facetgate_sha256"],
        "facetgate_validation_threshold_sha256":FACET[seed]["validation_threshold_sha256"],
        "facetgate_feature_cache_sha256":FACET[seed]["feature_cache_sha256"],
        "facetgate_feature_cache_manifest_sha256":FACET[seed]["feature_cache_manifest_sha256"],
        "facetgate_result_sha256":FACET[seed]["facet_result_sha256"],
        "development_verdict":FACET[seed]["facet_result"].get("verdict"),
        "development_test_forwards":FACET[seed]["facet_result"].get("locked_test_model_forwards"),
        "execution_mode":FACET[seed]["run_config"].get("execution_mode"),
    }
atomic_json(WORK_ROOT/"FINAL_PRETEST_ASSET_FREEZE.json",freeze)
print("[FIVE-SEED GEMMA4 FACETGATE ASSET FREEZE PASS]")
display(pd.DataFrame([
    {"seed":s,"gated_step":EXPECTED_GATED_STEP[s],"facetgate_step":EXPECTED_FACETGATE_STEP[s],
     "dev_verdict":FACET[s]["facet_result"].get("verdict"),
     "parent_sha":GATED[s]["checkpoint_content_sha256"][:12],
     "facet_sha":FACET[s]["selected_facetgate_sha256"][:12]}
    for s in PUBLICATION_SEEDS
]))


In [ ]:
# Cell 4 — Canonical 9,318-item dataset and ordered TEST audit for Gemma4
# No model forward occurs here.
import ast, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def read_table(path):
    p = Path(path)
    if p.suffix.lower()==".csv":
        return pd.read_csv(p)
    if p.suffix.lower() in {".jsonl",".ndjson"}:
        return pd.read_json(p, lines=True)
    obj = json.loads(p.read_text(encoding="utf-8"))
    if isinstance(obj,list):
        return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in ["data","records","items","annotations","samples"]:
            if isinstance(obj.get(k),list):
                return pd.DataFrame(obj[k])
        if obj and all(isinstance(v,dict) for v in obj.values()):
            return pd.DataFrame([{"__dict_key__":k,**v} for k,v in obj.items()])
    raise ValueError(f"Unsupported table structure: {p}")

def resolve_col(df,cands,what,required=True):
    low={str(c).casefold():c for c in df.columns}
    for c in cands:
        if c.casefold() in low:
            return low[c.casefold()]
    if required:
        raise KeyError(f"Cannot resolve {what}; columns={list(df.columns)}")
    return None

def image_name(v):
    return str(v).replace("\\","/").split("/")[-1]

def norm_key(v):
    s=image_name(v).strip().casefold()
    return re.sub(r"\.(jpg|jpeg|png|webp|bmp)$","",s)

def norm_binary(v,task):
    if isinstance(v,(bool,np.bool_)):
        return "Yes" if bool(v) else "No"
    s=str(v).strip().casefold()
    if s in {"yes","y","1","true","positive","pos"}:
        return "Yes"
    if s in {"no","n","0","false","negative","neg"}:
        return "No"
    raise ValueError(f"{task}: bad binary label {v!r}")

def norm_language(v):
    s=" ".join(str(v).strip().split()).casefold()
    aliases={
        "bangla":"Bangla","bengali":"Bangla","english":"English",
        "banglish":"Banglish","banglish (bangla + english)":"Banglish",
        "banglish(bangla + english)":"Banglish","bangla + english":"Banglish",
        "bangla+english":"Banglish","bengali + english":"Banglish",
        "romanized bangla":"Banglish","romanised bangla":"Banglish",
    }
    if s not in aliases:
        raise ValueError(f"bad language {v!r}")
    return aliases[s]

def norm_target(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        vals=[]
    elif isinstance(v,(list,tuple,set)):
        vals=list(v)
    elif isinstance(v,str):
        s=v.strip()
        if not s:
            vals=[]
        else:
            try:
                q=json.loads(s); vals=q if isinstance(q,list) else [q]
            except Exception:
                try:
                    q=ast.literal_eval(s); vals=list(q) if isinstance(q,(list,tuple,set)) else [q]
                except Exception:
                    vals=re.split(r"[,;|]",s)
    else:
        vals=[v]
    canon={x.casefold():x for x in TARGET_LABELS}
    canon["other"]="Others"
    out=[]
    for x in vals:
        sx=str(x).strip()
        if not sx or sx.casefold() in {"none","no target","null","nan"}:
            continue
        if sx.casefold() not in canon:
            raise ValueError(f"bad target {x!r}")
        c=canon[sx.casefold()]
        if c not in out:
            out.append(c)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(df):
    rows=[]; errors=[]
    for rn,r in df.iterrows():
        try:
            data=r.get("data",{})
            if isinstance(data,str):
                data=json.loads(data)
            image_value=None
            if isinstance(data,dict):
                for k in ["image","image_id","image_name","filename","file_name","source_image_name"]:
                    if data.get(k) not in [None,""]:
                        image_value=data[k]; break
            if image_value is None:
                for k in ["image","image_id","image_name","filename","file_name","source_image_name","id"]:
                    if k in r and pd.notna(r[k]):
                        image_value=r[k]; break
            anns=r.get("annotations",[])
            if isinstance(anns,str):
                anns=json.loads(anns)
            if not isinstance(anns,list) or not anns:
                raise ValueError("no annotations")
            coll={k:[] for k in BINARY_TASKS+["language","target"]}
            for ann in anns:
                for res in (ann.get("result",[]) if isinstance(ann,dict) else []):
                    if not isinstance(res,dict):
                        continue
                    frm=str(res.get("from_name","")).casefold()
                    val=res.get("value",{}) or {}
                    choices=val.get("choices",val.get("labels",[]))
                    if isinstance(choices,str):
                        choices=[choices]
                    task=next((k for k in coll if k in frm),None)
                    if task and choices:
                        coll[task].append(list(choices))
            item={"record_key":norm_key(image_value),"label_source_id":str(image_value)}
            for t in BINARY_TASKS:
                vals=[norm_binary(b[0],t) for b in coll[t] if len(b)==1]
                if not vals or len(set(vals))!=1:
                    raise ValueError(f"{t} non-unanimous/missing {vals}")
                item[t]=vals[0]
            lv=[norm_language(b[0]) for b in coll["language"] if len(b)==1]
            if not lv or len(set(lv))!=1:
                raise ValueError(f"language non-unanimous/missing {lv}")
            item["language"]=lv[0]
            tv=[tuple(norm_target(b)) for b in coll["target"] if b]
            if not tv:
                tv=[tuple()]
            if len(set(tv))!=1:
                raise ValueError(f"target non-unanimous {tv}")
            item["target_list"]=list(tv[0])
            rows.append(item)
        except Exception as e:
            errors.append({"row":int(rn),"error":repr(e)})
    if errors:
        raise RuntimeError(f"Label parse errors={len(errors)} first={errors[:3]}")
    return pd.DataFrame(rows)

def parse_labels(path):
    df=read_table(path).copy()
    if "annotations" in df.columns:
        out=parse_labelstudio(df)
    else:
        idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"label id")
        out=pd.DataFrame({"record_key":df[idc].map(norm_key),"label_source_id":df[idc].astype(str)})
        for t in BINARY_TASKS:
            c=resolve_col(df,[t],t)
            out[t]=df[c].map(lambda x:norm_binary(x,t))
        lc=resolve_col(df,["language","lang"],"language")
        out["language"]=df[lc].map(norm_language)
        tc=resolve_col(df,["target","targets"],"target")
        out["target_list"]=df[tc].map(norm_target)
    if out.record_key.duplicated().any():
        raise RuntimeError("duplicate label IDs")
    return out

def parse_ocr(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"OCR id")
    tc=resolve_col(df,["cleaned_ocr","ocr_cleaned","ocr","text","ocr_text","clean_text"],"OCR text")
    out=pd.DataFrame({"record_key":df[idc].map(norm_key),"cleaned_ocr":df[tc].fillna("").astype(str)})
    if out.record_key.duplicated().any():
        raise RuntimeError("duplicate OCR IDs")
    return out

def parse_manifest(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image","image_name","filename","file_name","source_image_name"],"manifest id")
    sc=resolve_col(df,["split","set","partition","dataset_split"],"split")
    ic=resolve_col(df,["official_row_index","row_index","index"],"row index",False)
    out=pd.DataFrame({
        "record_key":df[idc].map(norm_key),
        "image_id":df[idc].map(image_name),
        "split":df[sc].astype(str).str.strip().str.casefold().replace({"val":"validation","valid":"validation","dev":"validation"}),
        "official_row_index":np.arange(len(df),dtype=int) if ic is None else pd.to_numeric(df[ic],errors="raise").astype(int)
    })
    if out.record_key.duplicated().any() or out.official_row_index.duplicated().any():
        raise RuntimeError("manifest IDs/indices not unique")
    return out.sort_values("official_row_index",kind="stable").reset_index(drop=True)

# Discover exact metadata by immutable SHA.
metadata=[]
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.lower() in {".json",".jsonl",".ndjson",".csv"} and p.stat().st_size<250_000_000:
            metadata.append(p)
    except OSError:
        pass

wanted={
    "labels":EXPECTED_LABEL_SHA256,
    "ocr":EXPECTED_OCR_SHA256,
    "split":EXPECTED_SPLIT_SHA256,
}
found={}
for i,p in enumerate(metadata,1):
    h=sha256_file(p)
    for role,expected in wanted.items():
        if h==expected:
            found[role]=p
    if i%100==0:
        print(f"[META SHA] {i}/{len(metadata)}",flush=True)

if set(found)!=set(wanted):
    raise FileNotFoundError(
        f"Canonical metadata not found by SHA. Found={found}. "
        "Add the canonical M3Offend dataset as an input."
    )

labels=parse_labels(found["labels"])
ocr=parse_ocr(found["ocr"])
manifest=parse_manifest(found["split"])
master=manifest.merge(labels,on="record_key",validate="one_to_one").merge(ocr,on="record_key",validate="one_to_one")

if len(master)!=9318:
    raise RuntimeError(f"aligned rows={len(master)} != 9318")
counts={k:int(v) for k,v in master.split.value_counts().to_dict().items()}
if counts!={"train":7454,"validation":932,"test":932}:
    raise RuntimeError(f"split-count drift {counts}")

payload="\n".join(
    f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}"
    for r in master.sort_values("official_row_index").itertuples()
)+"\n"
row_sha=hashlib.sha256(payload.encode()).hexdigest()
if row_sha!=EXPECTED_CANONICAL_ROW_SHA256:
    raise RuntimeError(f"canonical-row SHA drift {row_sha}")

test_payload=[]
for r in master[master.split.eq("test")].sort_values("official_row_index").to_dict("records"):
    tar=[x for x in TARGET_LABELS if x in set(r["target_list"])]
    test_payload.append({
        "sample_id":str(int(r["official_row_index"])),
        "image":Path(str(r["image_id"])).name.lower(),
        "offensive":r["offensive"],"vulgar":r["vulgar"],"abuse":r["abuse"],"sarcastic":r["sarcastic"],
        "target":tar if tar else ["None"],
        "language":r["language"],
    })
test_sha=hashlib.sha256(
    json.dumps(test_payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
if test_sha!=EXPECTED_TEST_IDENTITY_SHA256:
    raise RuntimeError(f"ordered TEST identity SHA drift {test_sha}")

# Resolve every canonical image and verify full image-bundle SHA before TEST.
valid_ext={".jpg",".jpeg",".png",".webp",".bmp"}
by_key={}
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.casefold() in valid_ext:
            by_key.setdefault(norm_key(p.name),[]).append(p)
    except OSError:
        pass

image_map={}
for key,name in zip(master.record_key,master.image_id):
    cand=by_key.get(key,[])
    exact=[p for p in cand if p.name.casefold()==Path(str(name)).name.casefold()]
    chosen=exact[0] if len(exact)==1 else (cand[0] if len(cand)==1 else None)
    if chosen is None:
        raise RuntimeError(f"missing/ambiguous image {name}: {[str(x) for x in cand[:5]]}")
    image_map[key]=str(chosen)
master["image_path"]=master.record_key.map(image_map)

h=hashlib.sha256()
req=master.sort_values("image_id")[["image_id","image_path"]].drop_duplicates().to_dict("records")
for i,r in enumerate(req,1):
    name=Path(str(r["image_id"])).name
    h.update(name.encode()+b"\0"+sha256_file(r["image_path"]).encode()+b"\n")
    if i%500==0 or i==len(req):
        print(f"[IMAGE SHA] {i}/{len(req)}",flush=True)
image_sha=h.hexdigest()
if image_sha!=EXPECTED_IMAGE_BUNDLE_SHA256:
    raise RuntimeError(f"image-bundle SHA drift {image_sha}")

master["target_json"]=master.target_list.map(lambda x:json.dumps(x,separators=(",",":")))

# Locked-worker schema contract.
# `cleaned_ocr` is canonical. `ocr` is an exact compatibility alias retained because
# the original Gemma4 development feature worker used row["ocr"].
master["cleaned_ocr"]=master["cleaned_ocr"].fillna("").astype(str)
master["ocr"]=master["cleaned_ocr"]
if not master["ocr"].equals(master["cleaned_ocr"]):
    raise RuntimeError("OCR alias drift before locked TEST manifest write")

PREPARED_MANIFEST=WORK_ROOT/"prepared_manifest_FINAL_TEST.parquet"
_LOCKED_MANIFEST_COLUMNS=[
    "official_row_index","image_id","image_path","split","cleaned_ocr","ocr",
    *BINARY_TASKS,"language","target_json"
]
master[_LOCKED_MANIFEST_COLUMNS].to_parquet(PREPARED_MANIFEST,index=False)

# Read back the exact materialized parquet before any TEST model forward.
_manifest_probe=pd.read_parquet(PREPARED_MANIFEST)
if list(_manifest_probe.columns)!=_LOCKED_MANIFEST_COLUMNS:
    raise RuntimeError(f"Prepared manifest schema drift: {_manifest_probe.columns.tolist()}")
if not _manifest_probe["ocr"].equals(_manifest_probe["cleaned_ocr"]):
    raise RuntimeError("Prepared manifest OCR alias mismatch")
if _manifest_probe["image_path"].isna().any() or _manifest_probe["language"].isna().any():
    raise RuntimeError("Prepared manifest contains null image/language values")
if set(_manifest_probe["language"].astype(str)) - set(LANGUAGE_LABELS):
    raise RuntimeError("Prepared manifest contains unexpected language label")
for _task in BINARY_TASKS:
    if set(_manifest_probe[_task].astype(str)) - {"Yes","No"}:
        raise RuntimeError(f"Prepared manifest contains invalid binary label for {_task}")
for _s in _manifest_probe["target_json"].astype(str):
    _ = json.loads(_s)
print("[LOCKED TEST MANIFEST SCHEMA PASS]",{
    "columns":_LOCKED_MANIFEST_COLUMNS,
    "rows":len(_manifest_probe),
    "test_rows":int((_manifest_probe["split"].astype(str)=="test").sum()),
    "ocr_alias_exact":True,
    "TEST_model_forwards":0,
})
del _manifest_probe

print("[CANONICAL DATASET + TEST IDENTITY PASS]", {
    "rows":9318,
    "split_counts":counts,
    "ordered_test_sha256":test_sha,
    "image_bundle_sha256":image_sha,
    "TEST_model_forwards":0,
})


In [ ]:
# Cell 5 — Validation replay of all five frozen Gemma4 FacetGate packages
# Replays the small FacetGate specialist on CUDA:0, matching the development finalization device.
# TEST is still unopened.
import numpy as np, pandas as pd, torch, math
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, f1_score, average_precision_score, hamming_loss,
    precision_score, recall_score
)

# Development finalization evaluated the selected FacetGate specialist on cuda:0.
# Replaying it on CPU changes floating-point ordering of near-tied probabilities;
# mAP is rank-sensitive, so a CPU replay can produce a false mismatch even when the
# checkpoint, validation cache, and thresholds are byte-identical.
if not torch.cuda.is_available() or torch.cuda.device_count()<1:
    raise RuntimeError("Gemma4 validation replay requires CUDA:0 to match the frozen development execution path.")
REPLAY_DEVICE=torch.device("cuda:0")
DISCRETE_REPLAY_ABS_TOL=2e-5
MAP_REPLAY_ABS_TOL=1e-3  # 0.10 percentage point; artifact SHAs remain the primary integrity control.
print("[VALIDATION REPLAY DEVICE]",{"device":str(REPLAY_DEVICE),"map_abs_tol":MAP_REPLAY_ABS_TOL,
                                    "TEST_model_forwards_before_open":0})

class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden),nn.Dropout(dropout),
        )
    def forward(self,x): return x+self.block(self.norm(x))

class FacetGateTarget(nn.Module):
    def __init__(self):
        super().__init__(); D=1024
        self.product_norm=nn.LayerNorm(D); self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))
    def forward(self,v,t,J,zG):
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t); diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],1)
        vw=F.softmax(self.target_view_gate(torch.cat([v,t,prod,diff],-1).float()),-1)
        h=(views.float()*vw.unsqueeze(-1)).sum(1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        dit=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),
                        dit.unsqueeze(-1).expand(-1,7)],-1)
        slogit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        sel=torch.sigmoid(slogit)
        zstar=zG.float()+sel*(zH.float()-zG.float())
        return zstar,{"zH":zH,"selector":sel,"selector_logit":slogit,"view_weights":vw,
                      "image_text_disagreement":dit}

def target_metric_bundle(y,p,thr,full=False):
    y=np.asarray(y,int); p=np.asarray(p,float)
    th=np.asarray(thr,float).reshape(1,-1)
    pred=(p>=th).astype(int)
    f=[float(f1_score(y[:,j],pred[:,j],zero_division=0)) for j in range(7)]
    ap=[float(average_precision_score(y[:,j],p[:,j])) if len(np.unique(y[:,j]))>1 else float("nan") for j in range(7)]
    detail={}
    if full:
        for j,lab in enumerate(TARGET_LABELS):
            detail[lab]={
                "support":int(y[:,j].sum()),
                "precision":float(precision_score(y[:,j],pred[:,j],zero_division=0)),
                "recall":float(recall_score(y[:,j],pred[:,j],zero_division=0)),
                "f1":f[j],"ap":ap[j],"threshold":float(th[0,j]),
            }
    none_gold=(y.sum(1)==0).astype(int); none_pred=(pred.sum(1)==0).astype(int)
    out={
        "macro_f1":float(np.mean(f)),
        "map":float(np.nanmean(ap)),
        "micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
        "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
        "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
        "hamming_loss":float(hamming_loss(y,pred)),
        "per_label_f1":dict(zip(TARGET_LABELS,f)),
        "per_label_ap":dict(zip(TARGET_LABELS,ap)),
        "rare_target_macro_f1":float(np.mean([f[TARGET_LABELS.index("National")],f[TARGET_LABELS.index("Religious")]])),
        "none_support":int(none_gold.sum()),
        "none_precision":float(precision_score(none_gold,none_pred,zero_division=0)),
        "none_recall":float(recall_score(none_gold,none_pred,zero_division=0)),
        "none_f1":float(f1_score(none_gold,none_pred,zero_division=0)),
    }
    if full: out["per_label"]=detail
    return out,pred

def load_facet_state(seed,device=REPLAY_DEVICE):
    state=torch.load(FACET[seed]["selected_facetgate_checkpoint"],map_location="cpu",weights_only=False)
    if int(state.get("seed",-1))!=seed or int(state.get("step",-1))!=EXPECTED_FACETGATE_STEP[seed]:
        raise RuntimeError(f"Seed {seed} selected FacetGate state drift")
    if state.get("protocol_sha256")!=FACETGATE_PROTOCOL_SHA256:
        raise RuntimeError(f"Seed {seed} selected FacetGate protocol drift")
    sd=state.get("state_dict",state.get("facetgate_state"))
    if not isinstance(sd,dict): raise RuntimeError(f"Seed {seed} selected FacetGate state_dict missing")
    model=FacetGateTarget().to(device).eval()
    model.load_state_dict(sd,strict=True)
    return model,state

master_val=master[master.split.eq("validation")].sort_values("official_row_index").reset_index(drop=True)
expected_val_idx=master_val.official_row_index.to_numpy(dtype=np.int64)
val_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in master_val.target_json
],dtype=np.int64)

dev_rows=[]
for seed in PUBLICATION_SEEDS:
    if sha256_file(FACET[seed]["feature_cache"])!=FACET[seed]["feature_cache_sha256"]:
        raise RuntimeError(f"Seed {seed} feature-cache SHA changed after pretest freeze")
    fc=np.load(FACET[seed]["feature_cache"],allow_pickle=False)
    split=fc["split"].astype(str)
    vm=split=="validation"
    if int(vm.sum())!=932: raise RuntimeError(f"Seed {seed} validation cache rows={vm.sum()}")
    idx=fc["official_row_index"][vm].astype(np.int64)
    if not np.array_equal(idx,expected_val_idx):
        raise RuntimeError(f"Seed {seed} validation row identity/order drift")
    V=fc["v"][vm].astype(np.float32); T=fc["t"][vm].astype(np.float32)
    J=fc["J"][vm].astype(np.float32); ZG=fc["zG"][vm].astype(np.float32)
    Y=fc["y_target"][vm].astype(np.int64)
    if not np.array_equal(Y,val_gold_target):
        raise RuntimeError(f"Seed {seed} validation Target gold drift")
    model,state=load_facet_state(seed,REPLAY_DEVICE)
    pf=[]; sels=[]; views=[]
    with torch.inference_mode():
        for st in range(0,932,128):
            sl=slice(st,min(st+128,932))
            vv=torch.from_numpy(V[sl]).to(REPLAY_DEVICE)
            tt=torch.from_numpy(T[sl]).to(REPLAY_DEVICE)
            jj=torch.from_numpy(J[sl]).to(REPLAY_DEVICE)
            zg=torch.from_numpy(ZG[sl]).to(REPLAY_DEVICE)
            zs,ex=model(vv,tt,jj,zg)
            pf.append(torch.sigmoid(zs).cpu().numpy())
            sels.append(ex["selector"].cpu().numpy())
            views.append(ex["view_weights"].cpu().numpy())
    PF=np.concatenate(pf)
    th=np.asarray([FACET[seed]["facetgate_thresholds"]["facetgate_target"][lab] for lab in TARGET_LABELS],float)
    fm,_=target_metric_bundle(Y,PF,th)
    archived=FACET[seed]["facet_result"].get("target_facetgate_calibrated",{})
    # Threshold-dependent metrics must reproduce the frozen development result.
    discrete_keys=["macro_f1","micro_f1","samples_f1","exact_set_accuracy","hamming_loss","rare_target_macro_f1"]
    discrete_diffs={}
    for k in discrete_keys:
        if k in archived:
            d=abs(float(fm[k])-float(archived[k])); discrete_diffs[k]=d
            if d>DISCRETE_REPLAY_ABS_TOL:
                raise RuntimeError(
                    f"Seed {seed} validation replay mismatch {k}: replay={fm[k]} archived={archived[k]} diff={d}"
                )
    # mAP is threshold-independent but rank-sensitive. The immutable checkpoint/cache/threshold
    # SHAs above are the primary integrity proof. Use same-device CUDA replay plus a narrow
    # numerical tolerance to avoid false failure from near-tied score ordering.
    map_abs_diff=abs(float(fm["map"])-float(archived.get("map",fm["map"])))
    if map_abs_diff>MAP_REPLAY_ABS_TOL:
        raise RuntimeError(
            f"Seed {seed} validation replay mAP drift is too large: "
            f"replay={fm['map']} archived={archived.get('map')} diff={map_abs_diff} tol={MAP_REPLAY_ABS_TOL}"
        )
    if map_abs_diff>DISCRETE_REPLAY_ABS_TOL:
        print("[VALIDATION mAP NUMERICAL DRIFT ACCEPTED]",{
            "seed":seed,"replay_map":float(fm["map"]),"archived_map":float(archived["map"]),
            "abs_diff":map_abs_diff,"reason":"rank-sensitive near-tie ordering; immutable artifact SHAs matched",
            "TEST_model_forwards_before_open":0
        })
    delta=float(FACET[seed]["facet_result"].get("delta",{}).get("target_macro_f1_pp",
                100*(float(fm["macro_f1"])-float(FACET[seed]["facet_result"]["target_gated_calibrated"]["macro_f1"]))))
    dev_rows.append({
        "seed":seed,"gated_step":EXPECTED_GATED_STEP[seed],"facetgate_step":EXPECTED_FACETGATE_STEP[seed],
        "validation_target_macro_f1":float(fm["macro_f1"]),
        "validation_target_map_replay":float(fm["map"]),
        "validation_target_map_archived":float(archived.get("map",fm["map"])),
        "validation_target_map_abs_diff":map_abs_diff,
        "validation_target_delta_pp":delta,
        "verdict":FACET[seed]["facet_result"].get("verdict"),
        "replay_device":str(REPLAY_DEVICE),
    })
    print("[VALIDATION REPLAY PASS]",dev_rows[-1])
    del model,state,fc,V,T,J,ZG,Y,PF
    torch.cuda.empty_cache()

dev_df=pd.DataFrame(dev_rows).sort_values("seed").reset_index(drop=True)
mean_delta=float(dev_df.validation_target_delta_pp.mean())
positive=int((dev_df.validation_target_delta_pp>0).sum())
if not all(str(x).upper()=="PASS" for x in dev_df.verdict):
    raise RuntimeError("Not all Gemma4 FacetGate development seeds are PASS")
if mean_delta<1.0 or positive<4:
    raise RuntimeError(f"Five-seed promotion rule not satisfied mean_delta={mean_delta} positive={positive}")

gate={
    "mean_target_delta_pp":mean_delta,
    "positive_seed_count":positive,
    "all_five_development_verdicts_pass":True,
    "validation_replay_pass":True,
    "validation_replay_device":str(REPLAY_DEVICE),
    "map_replay_abs_tolerance":MAP_REPLAY_ABS_TOL,
    "promotion_decision_source":"frozen development artifacts; replay is integrity verification only",
    "TEST_model_forwards_before_open":0,
}
atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE_PASS.json",{"rows":dev_rows,"gate":gate})
print("[FIVE-SEED DEVELOPMENT REPLAY + PROMOTION PASS]",gate)
display(dev_df)


In [ ]:
# Cell 6 — Acquire the exact pinned Gemma4 snapshot once
# Still no TEST model forward.
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

MODEL_DIR=TEMP_ROOT/f"gemma4_12b_it_{MODEL_REVISION[:12]}"
MODEL_DIR.mkdir(parents=True,exist_ok=True)

# Reuse a mounted exact model.safetensors when available; otherwise download pinned revision.
attached=None
for p in INPUT_ROOT.rglob("model.safetensors"):
    try:
        if "gemma" in str(p).casefold() and sha256_file(p)==MODEL_WEIGHTS_SHA256:
            attached=p; break
    except Exception:
        pass

token=None
try: token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: token=None

if attached is not None:
    small=["*.json","*.jinja","*.txt","*.model","*.tiktoken","*.py","tokenizer*","processor*","preprocessor*","chat_template*"]
    snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=token,local_dir=str(MODEL_DIR),allow_patterns=small,max_workers=4)
    target=MODEL_DIR/"model.safetensors"
    if target.exists() or target.is_symlink(): target.unlink()
    os.symlink(str(attached),str(target))
else:
    if not token:
        raise RuntimeError("HF_TOKEN Kaggle secret is required unless exact Gemma4 model.safetensors is attached.")
    snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=token,local_dir=str(MODEL_DIR),max_workers=4)

weight=MODEL_DIR/"model.safetensors"
if not weight.is_file(): raise RuntimeError("Pinned Gemma4 model.safetensors missing")
if sha256_file(weight)!=MODEL_WEIGHTS_SHA256: raise RuntimeError("Pinned Gemma4 model weight SHA mismatch")
cfg=json.loads((MODEL_DIR/"config.json").read_text(encoding="utf-8"))
if cfg.get("model_type")!="gemma4_unified": raise RuntimeError(f"Unexpected model_type={cfg.get('model_type')}")
print("[PINNED GEMMA4 SNAPSHOT PASS]",{"path":str(MODEL_DIR),"revision":MODEL_REVISION,"TEST_model_forwards":0})


In [ ]:
# Cell 7 — Embedded exact Gemma4 locked-TEST feature worker
# Derived only by changing the allowed split from TRAIN+validation to locked TEST,
# adding explicit final-TEST authorization and TEST-forward accounting.
import base64,zlib,hashlib,re
DEV_FEATURE_WORKER_SHA256 = "7dd04c94a4e91767767b826be217f306f280b30040a165152fd5c63b74041d85"
LOCKED_TEST_FEATURE_WORKER_SHA256 = "891457cf71f5876bb8b265c92405232282dad1d01f2b0668afab13056c99ad55"
LOCKED_TEST_FEATURE_WORKER_ZLIB_B64 = 'eNrVPGtznEiS3/UrGCbuFjSIkWSf47a9XITGljSakSWHLN/dRG8Hi6BQs6KB4SF1W9f72y8fBVTRtCzNzl7cOhx2Q1VlZWXlu7KIy3xh+H7c1E0pfN9IFkVe1kaQZXkd1EmeVTvyVV45f63yzJkH1TxNbpw6WQinLoNQ3AThnVOvClHtxAiuCGrs0cL6CI8tkKxZFCsjqIyscIogi+AX/C0ip87LcM7/ullGPbLu0Y2bLERkghRbTniaj2fn7RRni+BWOPTvSZIKiYaI6w4H+P0hj0TKTYB2VsV5uRBl1XY5aur8Y5mHoqry0vkhqaujLPphVYvqXZ7Fya1zKhaL4PXnLIkTETGwbkb3/PLovX999fni3dH18Xv/7MPR6fEn77psxM67k1MPKeemeRBVFpLDyitXZPdJmWdT8/T4w4ej1/7J0bvj61MY7Z9fvvvZf3d5cXJ2as5stxRB5NdiWVu2vXN1dPGzl2S1BoGG+tfHn659bIdBb433x/959u7YYxJG4j4JhRWbYRMFk0fstDbtncvP1x6hAyhOzbypi6b2yzyvCcLRldoalHUSB2HtR0lJzeeXp0pzmt/Klp3ro6vT4+tPXppUNTfWQXkraj8NbkRa0eAfzi6Orn5RutwkWVCu/Dqo7rjH+dHFqQojDbLbBqjdQ9nBLv71pX/23ntcThIDNtRInKWRZIYARhNlUAuL4NjrnUjEBiBpLapbZ3f37sGeGEWJhDSnRD7j5Pjo+vPV8cx0sMvdgxOnTTWnLbRptB8nWVILv2qAD8qVtZwYTN1rkQHL2MbefxhREtaTHQP+LGPDM5ZA+ToI55ZN7xgAvOdxScUvrGXM7TXIXArNiNYydnENKW56P9YP8yarZRd+5QI+lu3Cr0XbN0OG7aZCmHvaeOqUxNo7xhr/3AdphbjHU26fuTFwbi3XgH+Cm2oRLKEPN+AAF94BFvB6gAtBXMCWaL3hxWi/IdRNeLD3YrKJiZxCg2BmQWbyqFKAfsuMx26gGaHCMidGVZdIa3q0nb69mgfUThwIHehZ7cDEgR4qFZV2Irw54Q1Q3nebA23db6Wdl+R3ffhZ6QAr7Vtx3WqbOvK+G8fsn2QVoJL69Ws/Ll4d+vdJBTrVB3UNHIpK/CbPBCCVuaDdmlQMONo0Tfqf9ODea0NqQuPg8AejqQRocuPNHw8O9x6SSBhl8LBHgI0gBoQECFv+V0FK3CUol5lx/doxfjg5eGMkldFkwX2QgGynAoxPZJx8hPdBGDaLJiU7ZIRBZuT3ooS9fTBuBAi7IEA0i59mh67xsxCFUc+F8XrvJqmNB5HczmvY4bxE82DcNDXqiFAYlxfnv8iBEQgvA8JpQeuDPViIKAHdAdtTLoI0+cII3MAGRyD4sKNGmC9AVwpUNScfXx06OCuiWLFsIQ7tYKBQKaomrQ0kMQ7G1gXaDyPKH2BPQMEvDGJAd4eGX8+BIvAXViyWImxw+u9JoyVhkO7BLt4kaVKvAHARJKVrnNUACnbg4vLaCOegKXlFUQD6VNSOwTrTMU5hVZERgPIBDgnR5DvGeX51BNuV3X1fhXkBL9K8gt10jLwAEw/4l8xD4VyEd0UOiseoRMpb6cCSwdKXAT+A/gVzWeNe13mYp67GN2hYQDZbRnPRJNwAfj7RwrJRI4FzEdQgkW0nxzD1bqZNGqCDQpCZkQE29KXhOBUMFYsbEUk2Nx3jAvrbre6TY4DK+LrXKGWQAPgrkGXwb47LMi8tky1/x/AqWNxIEBWEs0iqKsluXVA5CmMSfymo8TBATmnWcGv5+WD7IGgcHXL41JDDIQVU/CQZcAf7+UdeHv4mgklSsT4okkKkqBGqpgSHQkiGjTq6fQv8LwDkebAS5QVIkaQxyEMpjDrJViSq1TxvUvivDlYkg2T+O9FwdVq6dW6xE+SxY+SwwEn/iOzFq8MBNZ87SGJ9BioC9AfgdoNeo3EOiwzK1/DOAZjhHRBCaiSCgpSUqolfvJVwpG7xuVeYZ3WZg0WOxK9NAERGhbIIatCLIGciJNK67Z6C197JkLK/wAYaVNN+cv+6NnI8zONlAeIO025ZnsZJD0k915fw1rgFrB7xp4qTvTa7eeyhwLg6EVqPSZK8Xe3TK8WRflL5oAHV9Y7NonSFuchrp/55mdyCY5r6RV51qDA3u/iKNEGEQk+N6i5sDgV/z6cFQH+fFOmTmxCbnzPRUr6VIHVOyYyPmzOtW1FCq48ikQofpOMhKCMLdHcM/JgsReqDi9WISndjHbB/6GYDsAS1up9E1Yijq77olwG2DNwFVPJ5nlqtKuIpTX8hFiKPYwEclPnctfWpybhUoKFOwOsTtkSf5eFTDVb9YNIZ/70KZgXc7kWrUTrXorXHbjec2JGRDSDAQwstZZo23aOICOYVGVIp8nj+iSYBRKzqFWpXlW4v0Sntn3kSAefB4gMUIQ8taex2WspqZ1LWL1lKUhYVH3LXIIDQwNouOHnWYA3PkfTOiwQFvycDCMl4EPgl95wPAIeuFoqZmBjmKJTvDAp6I4j4K2sYO+kIOwZwUe3fiVUl460hsAFBniAjCfYA/j8LNVmDintwJYlvyaxJHfX/jMxgHv9JiQyY/1/T8lvjiBDQ4gmnDUrViOAPlRGX+RdggM3QYNuuaM+klEgL8WZtGikNr+MlkMloCjnTwG9rbUCQUk6vRyFMg0UBHbwRY+FSowVxqbdvu2me3SqZA9CcCQ6zNscZ33jG3oGNC9iKuttk1a+NEF+EBV37LMCSCCHVfInupAUxmVTOIyhySw9AzoVARiafyuU6NNHM2OVl2JR52VM0/JM7A9zVTrPztIBBL+SUrfK1sW0xmn8DPEKQkjzC3RoE21vn6sfI5Ypy67S/g0g/R5xVUeaFSYHu9qOV59EFj8n27ybX/ZOUXW3wjhKMutLfQrbEzLj7QdTzPLpGJ1j3yBw5wlZH675Sm7BhBSbTOCR+5KpuHTfmY8Eg8nF2tuTFZFIIWL6qMIck4fIL34dfWbAQvq8knRTj1Y1TvexnD+bwyFdTc60P2f6vu/o8oA1snTarJ5/tbThqoRDPszX22AQCPo82NMPUfB9pFqCAFgL4s7Js290K43A7jMOvwehtg68pKg3kZlhgt8lA2g7jSlRJ1ATpURQUMJXV5/0mXfDg+8g7vi99+ChZTDD17EjGlw9RmRd5U08476rIfNUUANh2OzC9AJH6QTUHDAkzd8G+BXMMe1FUBp0+QQgsQHcEqa42cDxFozi4xU0hmOxzenz+2dp8/Z6xt+QqNjtI2AyWiPASIEoopsdgSzUIZzlcgqpql2x1FLKWEBDxnlECj6z0O3xGM/2sjRvJ7mp7+II9a0EpqbyuQyRPJujEJm5Ik7QqMvki8NCmgzVX+1bzoBTRtr5I1C6rT/0lmTWAiJy09X1AOOQbjdVaRhnbZH7LLDOYBM/i/tFzyGSi36U2oePHVitYbHi/iDKvLIRSwuRJKSr/tgzaGHYcHuL+90O7DehsqV/SIThFEa5I7ydZYFTMB12ZA0ZpKUUbTPNcpZgzlDuFhnSOCMD4nBGkqj9ABOsgDyTNmY5CwIqwYiRYSt6DAbUeNWkHPTdUndXc6XkZ5/YlrAFHs6rUfQpMW9aYuCCcu6b1UOz4gHQugmgEwXqibAZic2D3gJkIQ4DdgSqC1DYTx6cik0enQ+bns9ytg+Tpr62ovaAo0pVPjlpSr3wpu1IztXlqZEw1S1fmN+0phyb5QzC+0lNVB+CtWuijMtLgeGbA/jblshXQf/KM/YGLyppYRcuhQzthKvooeOjiDQg3Igu5UYYb0mMahhiADw37kzq/PrWU0c4311SAEvpAWCWWWD3hB5WlOo+jawDJ3Hf3QQQkvG5N8lxkBMND6L/7YjwR7DYsaXe/RudNTE18be48Z2+2GNnd3SQDZ65SWItfAPZ4tGnJ5p7JCxEm8hQeWyAcKyxTvvXr/A6keQGSavZDFouuN/DnQnbinHJUKVoGPFhUa6TP2/7dO4Y6cyl12gMHFm8J3Zpbq11Um0LwcSUUrjh4EuuHAfj6Q8WN9haAgdxXta/GLSoD/L15262h4e8SDpLj03kdxsZKXh77PTveY8cCtwjIuFi44lcLVOy/9lvamx70DrSO+1pH+G39TfKT3Ox+mhiohOtr7fQYAHW/kNQ9am6QrdAF9g6eS27zqAatDXQ0kKBVsChSgccqAJjhGsTMlauw+w1p/r9161Rm1VgpiKhlgIRKHmaSh7kohQXdpxMO3GaOTgenH2QPFz+Gxe+w9qaiSgTaBaIAZ2U579Ai151YtqoH8QAvdiGCzFo6xmKAw7ec0AU3p+HcjEoAPA6MRJrcUNnSxLgILnb3oe8CLFeFT64Ga0nHEcuN0hzSR4DGK2SeLekyUnNBLAY7sCE6BMfZeA1Tb75UfUjFGC6lHRw/GnG2CBvTM8sXtAjMsh04xp0QBe4vSSknGX0sI+KU4RNnLtJq4IoZlm18z+D73btPWiU7jCAsdVPnjiJtqru5bIcPYoPh6J6RlZxoAu+X7BpyUpW90XGviXv3o2+7XayS20WesOEjD10SJQQGmfKwGUWvHjCDqnPAZHSzKh67dYuZzgQ0qHUAZnnPAPdpV5u789kVD94icGqirEBbix6q5mRP65nFIxRPVeux7mCk+W1Sa1BUbxgh4Szwv+0q/L7hAA/gKSHBrF2D5hQzVKWXvQGhDSS67e+9Yzm67bGROmQQjqEHFyZuHLGLOTFuXeIbu6si1AXGZF5kDQ3dFUPAfL5tHHFhN6xXoU+PkvzZsSQMla8GHbGqo/bvKXs42lTjrMvRpp+woA45aLT1yylW5A1o3/fkMrcoqcIyAdUQ4BkxWBA8o4YFFhYdsUxGkyhdrdsUHBxHO96edeVvV7xzeZausHRvr/NEeI7K+K+z6x8vP19jfYYoEzzqQT8ZT3ewSO6hBJkWVKSC2UWYzG1ro9jLe2QWRb7FPCmWjUCMjhxMmLv4MvKV5GApwqasxMapNYCbYmfkS+o+gHvTxEOo8Cp+Lkjo2wEM50ka+QyWfg8B08sSIkPVBY+VcYbnGWZLSlM3mVj9kmRN76PilOCsOZLkOBlBcnuiyoLRaugC4Api87GfeO0+Aqg1SS+DU3Pi0F0WDQeRP+ApmuwpdnKUTZ4QX8kiPMrNErP1VVRAcKBPTRHJk6wrK2DbohDUObUlR3NbflOJ8r5r65HgZiVyA68XlHUHa68byj2bTJ2GO3fA9zocumKyFjJsST/0JUVG77WlM/kQ6oJKPKxHot7anrQzeY/yx3RyuD9bK7N6j/1vbhwWGkkmokLxDVRhMXUDrmBL2Cn0nMnyXzyj5Maesmr7y8ObjYVLviZ4I+tHvD1i27FgJzZ7ImxdxLrb6rbT+GLW5oh7ptTTZDklCFUx+9Z4lxdYIFvspeAwp8CEKL8Cy9cS0Cx1uurdWH7D6qwLx5+l6OwhDKm8xoCM6zV7U6FEWLkKHPG4u6thhjkEfZr1uH5BAG4I6/c3CNpXClEn3SfmV3RAYyuUPAe9YwgwxLqpaXVsJAoIyIG86eqtkQGxS6zFTVsjQ5dNfE0pakv+TWr7Rap7VH1zak/ECTrMCpzvDNPV2a1qbtAi3k0xqchD7MkMXAreMQd+oMPYm1K5J4jgHVqDsq6QWduxa92ukM0Y0gjmdOQZt3LvQ0nh+nm8tVC+q+f7x9U5hzGqboQPfIZ3kna6iJqb2swNPMnkmc8dTQcHa3chMF/d9m9BwCA1YY1JXKSA9GqDsgxWVntshfcz1HtNS1pYUtHxdBYKiIDJ1NEy+BYFz195aJpwuP22RWaKEQbuHG0qj8FcIO01vpNp5RkjdNMg58QioFtroCMCC4RV4FlkmT9g7XpT+6DFJKryQIM0CnTyHvEUdm46pgw14FfYmW4OKvBVV/CORMTkC4I8/Lc35lo1pp60joNJ9nCR/NMeWsmv1H2ecAUQDzYqwGMRdAYWM1UTozV+a5mXCQu+i8Vjpry+WTev1tYuGqIlfvWNFxZP13PrGEkAVBOEaQvMwzSgi8q2g3JVgBAZx2OD5L8ZoR7SHp1LvBwxOimgMgDshyJrPbH/dIxgfqWIeojjNmiIqFgUEE9JHw242VMKMvBqhjnBK3uO+ZCXaWRODh0T2dycoAzjL9sxJWoTOvqX3K+VHSAWfo8F9xzwy8gAfeXaoO0E0iBVQkSMKh/14iN0MEnP+Qm0tQc7/Dhrm0ohjyjUDt3LmXrXCkwoDEVno151uPJp3FgTTJHmWJHvU3PQ1PO8xMp6c0JZ9AFknlkeLGhklwUVbLF/+nR5AR4PB358ZwgCVZGiVkQ2qI08pmCQavoDsFk0UMlP4947YwlpqSaxnTUgpgVb0wz8S0vt7goRwubsG8+8Ov786dg//u+zT9dnF6c+Xi99DyH0GLtKbWoAj90JDKzuc7yyALxU0YUuAVE5eQhmW61PupRm5t/m7C3glXlFxHdW4e2vjZDbXuBdJVSP0COJgaqoniThToIklVe6yCHPKvSEZTGivDOBRgb1X8Ik7CNsug7BIL9n3CVU2CXBN0wwqRuVSQxq2jX+EqYiyACRPCz/gjsTBuBg4cWqt8Zf6N0iWAEaCSWGU7qtHGQSpqDCOywQgv2VZ4iARFC5uqEJczCOj2YexwmdGwG3+OizLcGuyAwjW6CqABhkfzqkTGeXE1aOcmbd3qFFZtHtD881NEL4kkwQ0AachbRZZNWGHaJeXzFG5yQnBl4u7ujcGSPJKxJ+b5YI8LpXryYuizzMHpvuqAjeTanDzI2TNM0CC1SrG1Q118qVtos3X9LKoo4qobYN+IpiHl3S5bsr3kkIITiYoJviHXMY6sQ9ITdRSipAyB4efTwbDwUY6QlK/2dNmvaTym1VM5Pa6mnf+fD+tyDQTtpkd1n+kBndLH3wjKUKfWJVi51b5LDLAK1H8xeBjtVFbq6/GinHX8EO9BPVEj/iRGtOrrTnMciWHiJBVGIRm+FBGqlz04YIF3UsX+OwxqTUdhMwEFO0upPJ4Qz0GayL26jURFHN0nugkzzQdOw7QCQGsBQbYzoy8Hz+jrSKmP2Y3ka5PSdgbITd7G+812/efEWSu2tc0JVnQBsGOo5ufxr8VQCHL2t1gFshxmIA4VGZIr7HaKuk5EFlEVW0uBpZBOMKZBHrtyrBbcpPTyNqF94QRwd/2i9kr341Q3cboSU3dP/iEX8rSq3/AgPvufyOA3294dO7H48/HPkfr45Pzs9Of7z28cYxLGTftD3PPDAnWjG3KENEEO0aG5gKY+IM2AaL8LUw4DvyAfBiLwdBwCB0fHN0cn18pQDtzWiSgTYTEJC5fK8YDPE9AKxAZGph/CxWRBAjBhsM5r+trf4jX6gkh6fPHFCo5TMvbI/BOubB9e6+ea0dm1gmr2QP4/IUTftegSXDDjKhR24usqXXAwFu8MG3TyNPZxA9gZiUlRRQ+jwG4ehusp7tUNVA35G5GYV972A2HWPW2eAgZMOV9hSyTEc87dn4eM1lHocx8Kp1OC0j+K07yU6j9AQUeAP8R1xab394PLXTSn3oaR9GcdEqAlhBJVUwj+KaAxGx3JhCCYcfwD6Lykc3StGYCNWl4ye8XQ4xHxcvo5vpmSVVdFM/uu/qbX6JxaK8TZL5ePmUy15usht68mmMvNqXxa/NvgULY6K8wcp7BqwP1Iq+1XPqgzeMNeZmvM1vwPx9BHHknV6KFmSuxqN3/Z4RLmN4OfIi4yIowNfkGHHtoOEJiwardmDNoD8VyuMaZOrIHRQKyXbONnnd93I2locgnI1UgpNUPvXAsgm18uhZ35yQgS/O541WOHdpsi25OMXuZE74zBOoDOKjPndpIEtafJFTwWdYhYcC521t6UE4W6+C6oOpaG4EaPf+OSAphSkbUDaeSLE4wCzEl8hw7b1VDrE25PQrJ18O/XZMvrC2J2NAdZECPCyr35qi3xj1gsPEKFyt5Ni3FAbCSHLlXX6+/j7WYnY0F+yy+B8u3x+f+1fHR+9/Mfsx7kOJpV704SST3vy5+xoMGpjprKUd6ksPXQMX/5EIU96zbLLQGlSmJqAudQ2G7i33ckCV+VqtYMXC4BAoJjrJPX6oBUzcbT33uswIvkNjJ0/mZ7Y9NanQb1BGKJMBAzUaiRBIa0FXp7pLiiEa/bbyZ5JK/TNJTzl1zoGyfDoAou9euTm4GFbp9q6cjbEyXRHBV16yQF1zDzAt8+r0B6WEDFBfFLUXm4+0CKWOYf3n7B0beYzIJn/OHnkTSlex/erRFBCItoM2wJsyZNBHCBI2mf6fObJOUp7Te2YB7qa0O56e8TG0ir/KvwMDeYuxPO5Olcd94US3b23dhd4+s9eDUyak+NQEkqv9/AKEkyCAezjWdg9mMTdneE2joGLUO6f/jIdSseopJVh+mtwJLG3VGEhTIBiv6PjVCR0CDdgKlXfLSvQpgLZA9n+ohlWbAEMspApAUo6yqNdIAe3Mky/7QtqcDiXXOtkcOunB1Y8eu2FCSBabyv217kduEhPs6d3Mu39aozoQ7/s3qGla3rC7g4lHU73/D/ulPvJGVeaaTy/UaTZBaujRJ60UBMc+XtB58eS5Wbbz9W8ajHkyOl1kCVLEtsfa3SUcFFcQ9CTWoYksGtyBGPOTOS066nPLCI+yx6VLD8PqontzEk270qHZdL/77JgL3gz8S58OhIWbtdKzfqqnPsFPyrCfnpwAq4y6rvDw7Cm++FyXZk6AWSEopfOyKY+UVUvLTWBcqLbs8ykzB4aP10jCFF1YPNksY3suoqtRRA+Q0btPKDlLDFUxYyOP40bRhD3/903oPY7c5c1rq/9Y37Tsyuxmm0NlVZeKmHb8WLpKMsAeQWGt1Qgn/3Kwv+95+3hik3h9VLkhCJYJiu8WXBAtFI3QGU4c+oabEpKKNCgqvK/mlfhRMMtSvIc96VHY3796s7/vvJKaMG9q7+hq6MXI8LlS3Bk3K76weYO1VcG9+EKxCWKGNTiNEhlsipqn7udyPJzt9xHlu91FCNMdxaSAhA5hyTzaYLgy6h5HwOrDO+p/v9m3zQeO8netD69fOPwnffhPLxz+5VQfj3L/MgBSpgZgWkl7KbBWRIbglJrVlwBcjWK3eg52A/laaaipDLLajtwYk7WyPkSpq6l9FkrKoenTGSk+sJNfr3U5pYLixGdV9E0rvGw9F8soAcevbq9lyEQKuLfPlF/KUvLQelF4/XgX7blfNSCTS0t5zW++M13oLp1kHKkGMMMzwoRKf7zD4WGh1DUVLKpIwX5ZCMjp52oDO1B3qFRSCMFMB+w+tNs7CV4b5gv6oPl9H88YfV/mKutyNZGnjrIKMRRFbRzTf/jdhV6O228Cu/SBVR96WljpgUnYnf8FPupsBQ=='
raw=zlib.decompress(base64.b64decode(LOCKED_TEST_FEATURE_WORKER_ZLIB_B64))
if hashlib.sha256(raw).hexdigest()!=LOCKED_TEST_FEATURE_WORKER_SHA256:
    raise RuntimeError("Embedded Gemma4 locked-TEST worker SHA mismatch")
LOCKED_TEST_WORKER_SOURCE=raw.decode("utf-8")
compile(LOCKED_TEST_WORKER_SOURCE,"gemma4_facetgate_locked_test_worker.py","exec")
WORKER_PATH=WORK_ROOT/"gemma4_facetgate_locked_test_worker.py"
WORKER_PATH.write_text(LOCKED_TEST_WORKER_SOURCE,encoding="utf-8")
if 'split"].eq("test")' not in LOCKED_TEST_WORKER_SOURCE or 'isin(["train","validation"])' in LOCKED_TEST_WORKER_SOURCE:
    raise RuntimeError("Locked TEST worker split guard drift")
_worker_row_attrs=set(re.findall(r"\\br\\.([A-Za-z_][A-Za-z0-9_]*)",LOCKED_TEST_WORKER_SOURCE))
_manifest_attr_contract={"official_row_index","image_path","split","cleaned_ocr","language","target_json",*BINARY_TASKS}
_bad_worker_attrs=sorted(_worker_row_attrs-_manifest_attr_contract)
if _bad_worker_attrs:
    raise RuntimeError(f"Locked TEST worker row/manifest schema drift: {_bad_worker_attrs}")

_worker_parent_refs=set(re.findall(r'parent\[["\']([^"\']+)["\']\]',LOCKED_TEST_WORKER_SOURCE))
_expected_parent_schema={"path","manifest","manifest_payload","adapter","classifier_heads","thresholds",
                         "normalized_thresholds","run_config","gated_root","checkpoint_content_sha256",
                         "dedupe_key","historical_parent_test_artifacts_ignored"}
_missing_parent_contract=sorted(_worker_parent_refs-_expected_parent_schema)
if _missing_parent_contract:
    raise RuntimeError(f"Locked TEST worker/frozen-parent schema mismatch: missing={_missing_parent_contract}")
if 'parent["checkpoint"]' in LOCKED_TEST_WORKER_SOURCE or "parent['checkpoint']" in LOCKED_TEST_WORKER_SOURCE:
    raise RuntimeError("Forbidden legacy parent['checkpoint'] access remains in locked-TEST worker")
if "build_feature_meta(parent,len(work),\"0\"*64)" not in LOCKED_TEST_WORKER_SOURCE:
    raise RuntimeError("Worker schema preflight does not exercise final metadata builder")
print("[LOCKED-TEST NESTED PARENT + FINALIZATION CONTRACT PASS]",{
    "parent_keys_referenced":sorted(_worker_parent_refs),
    "missing_parent_keys":[],
    "late_metadata_builder_preflighted":True,
    "TEST_model_forwards":0,
})
if "trunc(r.cleaned_ocr)" not in LOCKED_TEST_WORKER_SOURCE:
    raise RuntimeError("Locked TEST worker is not bound to canonical cleaned_ocr")
print({
    "development_worker_sha256":DEV_FEATURE_WORKER_SHA256,
    "locked_test_worker_sha256":LOCKED_TEST_FEATURE_WORKER_SHA256,
    "compile":"PASS","TEST_model_forwards_before_execution":0
})


In [ ]:
# Cell 8 — Final locked TEST for all five Gemma4 FacetGate seeds
# Opens TEST only after Cells 3–7 pass. No training / no threshold selection.
import os, sys, time, json, math, subprocess, shutil, gc, re, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    balanced_accuracy_score, matthews_corrcoef, brier_score_loss
)

test_master=master[master.split.eq("test")].sort_values("official_row_index").reset_index(drop=True)
expected_test_idx=test_master.official_row_index.to_numpy(dtype=np.int64)
test_gold_bin=np.asarray([[1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS] for r in test_master.to_dict("records")],dtype=np.int64)
test_gold_lang=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in test_master.language],dtype=np.int64)
test_gold_target=np.asarray([[1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS] for s in test_master.target_json],dtype=np.int64)

def ece_binary(y,p,bins=15):
    y=np.asarray(y,int); p=np.asarray(p,float)
    edges=np.linspace(0,1,bins+1); out=0.0
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(p>=lo)&((p<hi) if hi<1 else (p<=hi))
        if m.any(): out += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(out)

def binary_detail(y,p,thr):
    y=np.asarray(y,int); p=np.asarray(p,float); pred=(p>=float(thr)).astype(int)
    out={
        "support_total":int(len(y)),"support_no":int((y==0).sum()),"support_yes":int((y==1).sum()),
        "threshold":float(thr),"accuracy":float(accuracy_score(y,pred)),
        "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "mcc":float(matthews_corrcoef(y,pred)),
        "brier":float(brier_score_loss(y,p)),"ece_15bin":ece_binary(y,p,15),
        "confusion_matrix":confusion_matrix(y,pred,labels=[0,1]).tolist(),
    }
    for val,name in [(0,"no"),(1,"yes")]:
        yy=(y==val).astype(int); pp=(pred==val).astype(int)
        out[f"precision_{name}"]=float(precision_score(yy,pp,zero_division=0))
        out[f"recall_{name}"]=float(recall_score(yy,pp,zero_division=0))
        out[f"f1_{name}"]=float(f1_score(yy,pp,zero_division=0))
    if len(np.unique(y))==2:
        out["auroc"]=float(roc_auc_score(y,p)); out["auprc"]=float(average_precision_score(y,p))
    else:
        out["auroc"]=None; out["auprc"]=None
    return out,pred

def language_detail(y,probs):
    y=np.asarray(y,int); probs=np.asarray(probs,float); pred=probs.argmax(1)
    per={}
    for i,lab in enumerate(LANGUAGE_LABELS):
        yy=(y==i).astype(int); pp=(pred==i).astype(int)
        per[lab]={
            "support":int(yy.sum()),
            "precision":float(precision_score(yy,pp,zero_division=0)),
            "recall":float(recall_score(yy,pp,zero_division=0)),
            "f1":float(f1_score(yy,pp,zero_division=0)),
        }
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "confusion_matrix":confusion_matrix(y,pred,labels=np.arange(3)).tolist(),
        "per_class":per,
    },pred

def load_test_shards(seed,feature_dir):
    parts=[]; total_forwards=0; metas=[]
    for rank in (0,1):
        npz=feature_dir/f"locked_test_features_rank{rank}.npz"
        meta_path=feature_dir/f"locked_test_features_rank{rank}.json"
        if not npz.is_file() or not meta_path.is_file():
            raise RuntimeError(f"Missing TEST shard seed={seed} rank={rank}")
        meta=json.loads(meta_path.read_text(encoding="utf-8"))
        if int(meta.get("rank",-1))!=rank or int(meta.get("world",-1))!=2: raise RuntimeError("rank/world drift")
        if int(meta.get("seed",-1))!=seed: raise RuntimeError("seed drift")
        if meta.get("model_id")!=MODEL_ID or meta.get("model_revision")!=MODEL_REVISION: raise RuntimeError("model identity drift")
        if meta.get("parent_content_sha256")!=GATED[seed]["checkpoint_content_sha256"]: raise RuntimeError("parent SHA drift")
        if meta.get("test_identity_sha256")!=EXPECTED_TEST_IDENTITY_SHA256: raise RuntimeError("TEST identity drift in shard")
        if not bool(meta.get("locked_test_authorized",False)): raise RuntimeError("TEST shard missing authorization")
        total_forwards += int(meta.get("test_model_forwards",-1))
        metas.append(meta)
        with np.load(npz,allow_pickle=False) as z: parts.append({k:z[k] for k in z.files})
    keys=["official_row_index","split","v","t","J","zG","z_binary","z_language","y_binary","y_language","y_target"]
    M={k:np.concatenate([p[k] for p in parts]) for k in keys}
    order=np.argsort(M["official_row_index"].astype(np.int64),kind="stable")
    for k in M: M[k]=M[k][order]
    if total_forwards!=932: raise RuntimeError(f"Seed {seed} TEST forward count {total_forwards} != 932")
    if len(M["official_row_index"])!=932 or len(np.unique(M["official_row_index"]))!=932: raise RuntimeError("TEST row loss/duplication")
    if not np.array_equal(M["official_row_index"].astype(np.int64),expected_test_idx): raise RuntimeError("ordered TEST identity mismatch")
    if not np.all(M["split"].astype(str)=="test"): raise RuntimeError("non-TEST row in locked feature cache")
    return M,total_forwards,metas

def _expected_rank_indices(rank):
    return expected_test_idx[rank::2]

def _validate_npz_payload(seed, rank, npz_path):
    req=["official_row_index","split","v","t","J","zG","z_binary","z_language","y_binary","y_language","y_target"]
    try:
        with np.load(npz_path,allow_pickle=False) as z:
            if sorted(z.files)!=sorted(req):
                return False,f"npz keys drift: {sorted(z.files)}"
            arr={k:z[k] for k in req}
        idx=arr["official_row_index"].astype(np.int64)
        exp=_expected_rank_indices(rank)
        if not np.array_equal(idx,exp): return False,"rank official_row_index drift"
        n=len(exp)
        expected_shapes={
            "v":(n,1024),"t":(n,1024),"J":(n,1024),"zG":(n,7),
            "z_binary":(n,4),"z_language":(n,3),
            "y_binary":(n,4),"y_language":(n,),"y_target":(n,7),
        }
        for k,sh in expected_shapes.items():
            if tuple(arr[k].shape)!=tuple(sh): return False,f"{k} shape {arr[k].shape} != {sh}"
        if not np.all(arr["split"].astype(str)=="test"): return False,"non-TEST rows"
        sl=slice(rank,None,2)
        if not np.array_equal(arr["y_binary"].astype(np.int64),test_gold_bin[sl]): return False,"binary gold drift"
        if not np.array_equal(arr["y_language"].astype(np.int64),test_gold_lang[sl]): return False,"language gold drift"
        if not np.array_equal(arr["y_target"].astype(np.int64),test_gold_target[sl]): return False,"target gold drift"
        for k in ["v","t","J","zG","z_binary","z_language"]:
            if not np.isfinite(arr[k]).all(): return False,f"nonfinite {k}"
        return True,"PASS"
    except Exception as exc:
        return False,repr(exc)

def _write_recovered_meta(seed,rank,npz_path,meta_path,source_cfg_sha):
    meta={
        "rank":rank,"world":2,"rows":466,"sha256":sha256_file(npz_path),
        "parent_checkpoint":GATED[seed]["path"],
        "parent_content_sha256":GATED[seed]["checkpoint_content_sha256"],
        "seed":seed,"model_id":MODEL_ID,"model_revision":MODEL_REVISION,
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        "locked_test_authorized":True,"test_model_forwards":466,
        "recovered_from_completed_orphan_npz":True,
        "recovery_source_config_sha256":source_cfg_sha,
        "recovery_build_id":LOCKED_TEST_BUILD_ID,
    }
    atomic_json(meta_path,meta)
    return meta

def _recover_orphan_shards_from_mounted_failed_output(seed, feature_dir):
    """
    Recover the exact 466+466 TEST feature NPZs from the known metadata-finalization
    failure. No TEST forward is repeated. Recovery is accepted only when the old
    worker config, parent SHA, model identity, TEST identity, row identities, gold
    labels, shapes and finite feature payload all match the current frozen state.
    """
    recovered=[]
    diagnostics=[]
    for cfgp in INPUT_ROOT.rglob("locked_test_worker_config.json"):
        try:
            old=json.loads(cfgp.read_text(encoding="utf-8"))
        except Exception:
            continue
        if int(old.get("seed",-1))!=seed: continue
        if old.get("model_id")!=MODEL_ID or old.get("model_revision")!=MODEL_REVISION: continue
        if old.get("execution_mode")!="REUSE_EXISTING_GATED": continue
        if not bool(old.get("allow_locked_test",False)): continue
        if old.get("test_identity_sha256")!=EXPECTED_TEST_IDENTITY_SHA256: continue
        op=old.get("parent") or {}
        if op.get("checkpoint_content_sha256")!=GATED[seed]["checkpoint_content_sha256"]: continue
        mp=op.get("manifest_payload") or {}
        if int(mp.get("seed",-1))!=seed or int(mp.get("global_step",-1))!=EXPECTED_GATED_STEP[seed]: continue
        cfg_sha=sha256_file(cfgp)
        # The failed output may preserve the original seed directory tree at any depth.
        roots=[cfgp.parent, cfgp.parent/"locked_test_features"]
        candidates=[]
        for root in roots:
            if root.exists():
                candidates.extend(root.rglob("locked_test_features_rank*.npz"))
        # de-duplicate exact paths
        uniq={str(p.resolve()):p for p in candidates}
        for rank in (0,1):
            dst_npz=feature_dir/f"locked_test_features_rank{rank}.npz"
            dst_meta=feature_dir/f"locked_test_features_rank{rank}.json"
            if dst_npz.is_file() and dst_meta.is_file():
                continue
            matches=[]
            for p in uniq.values():
                if p.name!=f"locked_test_features_rank{rank}.npz": continue
                ok,why=_validate_npz_payload(seed,rank,p)
                diagnostics.append({"seed":seed,"rank":rank,"path":str(p),"valid":ok,"reason":why})
                if ok: matches.append(p)
            # Require exactly one content identity, tolerate duplicate copies of same bytes.
            bysha={}
            for p in matches: bysha.setdefault(sha256_file(p),p)
            if len(bysha)==1:
                src_npz=next(iter(bysha.values()))
                feature_dir.mkdir(parents=True,exist_ok=True)
                shutil.copy2(src_npz,dst_npz)
                meta=_write_recovered_meta(seed,rank,dst_npz,dst_meta,cfg_sha)
                recovered.append(rank)
                print("[RECOVERED ORPHAN LOCKED-TEST FEATURE SHARD — NO REPEAT FORWARD]",{
                    "seed":seed,"rank":rank,"rows":466,"npz_sha256":meta["sha256"],
                    "source_config":str(cfgp),"TEST_model_forwards_this_session":0
                },flush=True)
        if len(recovered)==2:
            break
    return sorted(set(recovered)),diagnostics

def _feature_shard_complete(seed,rank,feature_dir):
    npz=feature_dir/f"locked_test_features_rank{rank}.npz"
    mp=feature_dir/f"locked_test_features_rank{rank}.json"
    if not npz.is_file() or not mp.is_file(): return False
    try:
        meta=json.loads(mp.read_text(encoding="utf-8"))
        if int(meta.get("rank",-1))!=rank or int(meta.get("world",-1))!=2: return False
        if int(meta.get("seed",-1))!=seed or int(meta.get("rows",-1))!=466: return False
        if int(meta.get("test_model_forwards",-1))!=466: return False
        if meta.get("model_id")!=MODEL_ID or meta.get("model_revision")!=MODEL_REVISION: return False
        if meta.get("parent_content_sha256")!=GATED[seed]["checkpoint_content_sha256"]: return False
        if meta.get("test_identity_sha256")!=EXPECTED_TEST_IDENTITY_SHA256: return False
        if not bool(meta.get("locked_test_authorized",False)): return False
        if sha256_file(npz)!=meta.get("sha256"): return False
        ok,_=_validate_npz_payload(seed,rank,npz)
        return bool(ok)
    except Exception:
        return False

def _emergency_progress_zip(seed,seed_dir,reason):
    out=WORK_ROOT/"Gemma4_FacetGate_5Seed_LOCKED_TEST_EMERGENCY_PROGRESS.zip"
    temp=WORK_ROOT/"EMERGENCY_PROGRESS_STAGE"
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir(parents=True)
    dst=temp/f"seed_{seed}"
    shutil.copytree(seed_dir,dst)
    atomic_json(temp/"EMERGENCY_PROGRESS_REASON.json",{
        "seed":seed,"reason":str(reason),"build_id":LOCKED_TEST_BUILD_ID,
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    })
    if out.exists(): out.unlink()
    with zipfile.ZipFile(out,"w",compression=zipfile.ZIP_DEFLATED) as zf:
        for p in temp.rglob("*"):
            if p.is_file(): zf.write(p,p.relative_to(temp))
    print("[EMERGENCY TECHNICAL PROGRESS PACKAGE]",{"path":str(out),"reason":str(reason)},flush=True)
    return out

def language_conditioned(seed,bp,bpred,facet_prob,facet_pred):
    rows=[]
    for lid,lang in enumerate(LANGUAGE_LABELS):
        m=(test_gold_lang==lid)
        if not m.any(): raise RuntimeError(f"No TEST support for language {lang}")
        task_acc=[]; task_f1=[]
        row={"seed":seed,"language":lang,"support":int(m.sum())}
        for k,t in enumerate(BINARY_TASKS):
            a=float(accuracy_score(test_gold_bin[m,k],bpred[t][m]))
            f=float(f1_score(test_gold_bin[m,k],bpred[t][m],average="macro",zero_division=0))
            row[f"{t}_accuracy"]=a; row[f"{t}_macro_f1"]=f
            task_acc.append(a); task_f1.append(f)
        row["four_task_accuracy"]=float(np.mean(task_acc))
        row["four_task_macro_f1"]=float(np.mean(task_f1))
        tm=target_metric_bundle(test_gold_target[m],facet_prob[m],
             np.asarray([FACET[seed]["facetgate_thresholds"]["facetgate_target"][x] for x in TARGET_LABELS]),full=False)[0]
        row["target_exact_set_accuracy"]=float(tm["exact_set_accuracy"])
        row["target_macro_f1"]=float(tm["macro_f1"])
        row["target_map"]=float(tm["map"])
        row["five_task_accuracy"]=float((sum(task_acc)+tm["exact_set_accuracy"])/5.0)
        row["five_task_macro_f1"]=float((sum(task_f1)+tm["macro_f1"])/5.0)
        rows.append(row)
    return rows

def write_seed_paper_tables(seed_dir,result,lang_rows):
    # Core taskwise
    pd.DataFrame([
        {"task":t,**result["gated"]["core5"]["tasks"][t]}
        for t in [*BINARY_TASKS,"language"]
    ]).to_csv(seed_dir/"core5_taskwise.csv",index=False)

    # Binary classwise
    br=[]
    for t in BINARY_TASKS:
        d=result["gated"]["core5"]["tasks"][t]
        for cls in ["no","yes"]:
            br.append({
                "task":t,"class":cls.title(),"support":d[f"support_{cls}"],
                "precision":d[f"precision_{cls}"],"recall":d[f"recall_{cls}"],"f1":d[f"f1_{cls}"],
            })
    pd.DataFrame(br).to_csv(seed_dir/"binary_classwise.csv",index=False)

    # Language classwise
    lr=[]
    for lab,d in result["gated"]["core5"]["tasks"]["language"]["per_class"].items():
        lr.append({"language":lab,**d})
    pd.DataFrame(lr).to_csv(seed_dir/"language_classwise.csv",index=False)

    # Target labelwise: matched Gated and final FacetGate side by side
    tr=[]
    gd=result["gated"]["target_calibrated"]["per_label"]
    fd=result["facetgate"]["target_calibrated"]["per_label"]
    for lab in TARGET_LABELS:
        g=gd[lab]; f=fd[lab]
        tr.append({
            "label":lab,"support":f["support"],
            "gated_precision":g["precision"],"gated_recall":g["recall"],"gated_f1":g["f1"],"gated_ap":g["ap"],"gated_threshold":g["threshold"],
            "facetgate_precision":f["precision"],"facetgate_recall":f["recall"],"facetgate_f1":f["f1"],"facetgate_ap":f["ap"],"facetgate_threshold":f["threshold"],
            "delta_f1_pp":100*(f["f1"]-g["f1"]),"delta_ap_pp":100*(f["ap"]-g["ap"]),
        })
    pd.DataFrame(tr).to_csv(seed_dir/"target_labelwise.csv",index=False)
    pd.DataFrame(lang_rows).to_csv(seed_dir/"language_conditioned.csv",index=False)

# Optional exact technical resume from a previously attached progress ZIP produced by
# the same fixed build. No prediction is reused unless the later per-seed lock_id and
# ledger hash checks pass exactly.
RESUME_SEED_CANDIDATES={s:[] for s in PUBLICATION_SEEDS}
_resume_root=WORK_ROOT/"MOUNTED_PROGRESS_IMPORTS"
if _resume_root.exists(): shutil.rmtree(_resume_root)
_resume_root.mkdir(parents=True,exist_ok=True)
for zi,zp in enumerate(INPUT_ROOT.rglob("Gemma4_FacetGate_5Seed_LOCKED_TEST_PROGRESS.zip")):
    try:
        dest=_resume_root/f"zip_{zi}"
        dest.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zp,"r") as zf: zf.extractall(dest)
        for s in PUBLICATION_SEEDS:
            for cand in dest.rglob(f"seed_{s}"):
                marker=cand/"LOCKED_TEST_ONCE_MARKER.json"
                metrics=cand/"locked_test_metrics.json"
                ledger=cand/"locked_test_prediction_ledger.parquet"
                if marker.is_file() and metrics.is_file() and ledger.is_file():
                    RESUME_SEED_CANDIDATES[s].append(cand)
    except Exception as exc:
        print("[PROGRESS ZIP IGNORED]",{"path":str(zp),"reason":repr(exc)})

PER_SEED_RESULTS=[]
ALL_LANG_ROWS=[]
NEW_TEST_FORWARDS_THIS_SESSION=0
FACET_INFER_DEVICE=REPLAY_DEVICE

for seed in PUBLICATION_SEEDS:
    print("\n"+"="*100)
    print(f"FINAL LOCKED TEST — GEMMA4 FACETGATE — SEED {seed}")
    print("="*100)
    seed_dir=WORK_ROOT/f"seed_{seed}"; feature_dir=seed_dir/"locked_test_features"
    seed_dir.mkdir(parents=True,exist_ok=True); feature_dir.mkdir(parents=True,exist_ok=True)

    lock_payload={
        "build_id":LOCKED_TEST_BUILD_ID,"seed":seed,
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        "model_id":MODEL_ID,"model_revision":MODEL_REVISION,
        "gated_step":EXPECTED_GATED_STEP[seed],
        "gated_checkpoint_content_sha256":GATED[seed]["checkpoint_content_sha256"],
        "facetgate_step":EXPECTED_FACETGATE_STEP[seed],
        "facetgate_checkpoint_sha256":FACET[seed]["selected_facetgate_sha256"],
        "facetgate_validation_threshold_sha256":FACET[seed]["validation_threshold_sha256"],
        "facetgate_protocol_sha256":FACETGATE_PROTOCOL_SHA256,
    }
    lock_id=json_sha(lock_payload)
    marker=seed_dir/"LOCKED_TEST_ONCE_MARKER.json"
    metrics_path=seed_dir/"locked_test_metrics.json"
    ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"

    # Import a completed seed from a mounted technical-progress ZIP only when the exact
    # current lock_id, TEST identity, and file hashes agree.
    if not marker.is_file():
        for cand in RESUME_SEED_CANDIDATES.get(seed,[]):
            try:
                old=json.loads((cand/"LOCKED_TEST_ONCE_MARKER.json").read_text(encoding="utf-8"))
                cand_ledger=cand/"locked_test_prediction_ledger.parquet"
                cand_metrics=cand/"locked_test_metrics.json"
                if (
                    old.get("lock_id")==lock_id
                    and int(old.get("test_rows",-1))==932
                    and old.get("test_identity_sha256")==EXPECTED_TEST_IDENTITY_SHA256
                    and sha256_file(cand_ledger)==old.get("ledger_sha256")
                    and sha256_file(cand_metrics)==old.get("metrics_sha256")
                ):
                    if seed_dir.exists(): shutil.rmtree(seed_dir)
                    shutil.copytree(cand,seed_dir)
                    feature_dir=seed_dir/"locked_test_features"
                    feature_dir.mkdir(parents=True,exist_ok=True)
                    marker=seed_dir/"LOCKED_TEST_ONCE_MARKER.json"
                    metrics_path=seed_dir/"locked_test_metrics.json"
                    ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"
                    print("[IMPORTED IDENTICAL COMPLETED SEED FROM MOUNTED PROGRESS]",seed)
                    break
            except Exception:
                continue

    reuse=False
    # Reuse only an exact completed seed whose lock-id and ledger hash match this build.
    if marker.is_file() and metrics_path.is_file() and ledger_path.is_file():
        old=json.loads(marker.read_text())
        if old.get("lock_id")==lock_id and int(old.get("test_rows",-1))==932 and sha256_file(ledger_path)==old.get("ledger_sha256"):
            reuse=True
            print("[REUSE IDENTICAL COMPLETED SEED] no TEST forward repeated",seed)

    if not reuse:
        cfg={
            "seed":seed,"model_id":MODEL_ID,"model_revision":MODEL_REVISION,
            "model_local_path":str(MODEL_DIR),
            "prepared_manifest":str(PREPARED_MANIFEST),
            "output_root":str(seed_dir),"artifact_dir":str(feature_dir),"log_dir":str(seed_dir/"logs"),
            "binary_tasks":BINARY_TASKS,"language_labels":LANGUAGE_LABELS,"target_labels":TARGET_LABELS,
            "max_ocr_tokens":384,"image_max_soft_tokens":140,
            "fusion_hidden_size":1024,"shared_hidden_size":1024,"task_adapter_hidden_size":256,
            "dropout":0.10,"modality_dropout_probability":0.0,
            "parent":GATED[seed],
            "execution_mode":"REUSE_EXISTING_GATED",
            "allow_locked_test":True,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        }
        # Fail-closed materialized worker/config contract audit before either GPU starts.
        _worker_cfg_refs=set(re.findall(r'CFG\[\"([^\"]+)\"\]',LOCKED_TEST_WORKER_SOURCE))
        _worker_cfg_refs.update(re.findall(r"CFG\['([^']+)'\]",LOCKED_TEST_WORKER_SOURCE))
        _worker_cfg_refs.update(re.findall(r'CFG\.get\(\s*[\"\']([^\"\']+)',LOCKED_TEST_WORKER_SOURCE))
        _missing_worker_cfg=sorted(k for k in _worker_cfg_refs if k not in cfg)
        if _missing_worker_cfg:
            raise RuntimeError(f"Locked TEST worker/config mismatch before TEST open: missing={_missing_worker_cfg}")
        if cfg.get("execution_mode")!="REUSE_EXISTING_GATED":
            raise RuntimeError("Locked TEST worker must execute against frozen REUSE_EXISTING_GATED state")
        print("[LOCKED-TEST WORKER CONFIG CONTRACT PASS]",{
            "seed":seed,"required_keys":len(_worker_cfg_refs),"missing":[],
            "execution_mode":cfg["execution_mode"],"TEST_model_forwards_before_worker":0
        })

        (seed_dir/"logs").mkdir(exist_ok=True)
        cfgp=seed_dir/"locked_test_worker_config.json"
        cfgp.write_text(json.dumps(cfg,indent=2,default=str),encoding="utf-8")

        # Nested parent-schema audit catches errors such as parent['checkpoint'] versus
        # the canonical GATED parent key parent['path'] BEFORE any TEST model forward.
        _parent_refs=set(re.findall(r'parent\[["\']([^"\']+)["\']\]',LOCKED_TEST_WORKER_SOURCE))
        _missing_nested=sorted(k for k in _parent_refs if k not in cfg["parent"])
        if _missing_nested:
            raise RuntimeError(f"Locked TEST nested parent-schema mismatch before TEST open: missing={_missing_nested}")
        print("[LOCKED-TEST NESTED PARENT CONTRACT PASS]",{
            "seed":seed,"referenced_parent_keys":sorted(_parent_refs),"missing":[],
            "TEST_model_forwards_before_worker":0
        },flush=True)

        # Recover the completed 466+466 feature shards from the prior metadata-only crash
        # if that failed Kaggle output is mounted as an input dataset.
        _recovered,_recovery_diag=_recover_orphan_shards_from_mounted_failed_output(seed,feature_dir)
        _ranks_to_run=[r for r in (0,1) if not _feature_shard_complete(seed,r,feature_dir)]
        if not _ranks_to_run:
            print("[LOCKED TEST FEATURE CACHE COMPLETE — NO BACKBONE FORWARD REPEATED]",{
                "seed":seed,"recovered_ranks":_recovered,"ranks_to_run":[],
                "TEST_model_forwards_this_session":0
            },flush=True)
        else:
            print("[LOCKED TEST RANK EXECUTION PLAN]",{
                "seed":seed,"recovered_ranks":_recovered,"ranks_to_run":_ranks_to_run,
                "new_TEST_forwards_planned":466*len(_ranks_to_run)
            },flush=True)

        # Execute the exact materialized worker in schema-only mode before either worker
        # is allowed to load the 12B model. This catches manifest/row/config integration
        # errors in seconds instead of after both GPUs finish loading Gemma4.
        for _rank in _ranks_to_run:
            _env=os.environ.copy()
            _env["GEMMA4_FACETGATE_LOCK_CONFIG"]=str(cfgp)
            _env["FACET_TEST_RANK"]=str(_rank)
            _env["CUDA_VISIBLE_DEVICES"]="0,1"
            _env["TOKENIZERS_PARALLELISM"]="false"
            _env["GEMMA4_LOCK_SCHEMA_PREFLIGHT_ONLY"]="1"
            _cp=subprocess.run([sys.executable,"-u",str(WORKER_PATH)],env=_env,
                               text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
            if _cp.returncode!=0 or "schema-preflight-pass" not in _cp.stdout:
                print(_cp.stdout[-12000:],flush=True)
                raise RuntimeError(f"Materialized locked-TEST worker schema preflight failed rank={_rank} rc={_cp.returncode}")
            print("[MATERIALIZED WORKER SCHEMA PREFLIGHT PASS]",{
                "seed":seed,"rank":_rank,"TEST_model_forwards":0
            },flush=True)

        procs={}
        for rank in _ranks_to_run:
            env=os.environ.copy()
            env["GEMMA4_FACETGATE_LOCK_CONFIG"]=str(cfgp)
            env["FACET_TEST_RANK"]=str(rank)
            env["CUDA_VISIBLE_DEVICES"]="0,1"
            env["TOKENIZERS_PARALLELISM"]="false"
            logp=seed_dir/"logs"/f"rank{rank}.log"
            fh=open(logp,"w",buffering=1)
            procs[rank]=(subprocess.Popen([sys.executable,"-u",str(WORKER_PATH)],env=env,stdout=fh,stderr=subprocess.STDOUT),fh)
        failures={}
        for rank,(proc,fh) in procs.items():
            rc=proc.wait(); fh.close()
            print("[LOCKED TEST WORKER EXIT]",{"seed":seed,"rank":rank,"returncode":rc},flush=True)
            if rc!=0: failures[rank]=rc
        if failures:
            for rank in failures:
                lp=seed_dir/"logs"/f"rank{rank}.log"
                if lp.is_file(): print(lp.read_text(errors="replace")[-12000:])
            _emergency_progress_zip(seed,seed_dir,f"worker_failure_{failures}")
            raise RuntimeError(f"Seed {seed} locked TEST worker failure={failures}")

        NEW_TEST_FORWARDS_THIS_SESSION += 466*len(_ranks_to_run)
        M,forwards,metas=load_test_shards(seed,feature_dir)
        V=M["v"].astype(np.float32); T=M["t"].astype(np.float32); J=M["J"].astype(np.float32)
        ZG=M["zG"].astype(np.float32); ZB=M["z_binary"].astype(np.float32); ZL=M["z_language"].astype(np.float32)
        if not np.array_equal(M["y_binary"].astype(np.int64),test_gold_bin): raise RuntimeError("binary gold drift")
        if not np.array_equal(M["y_language"].astype(np.int64),test_gold_lang): raise RuntimeError("language gold drift")
        if not np.array_equal(M["y_target"].astype(np.int64),test_gold_target): raise RuntimeError("target gold drift")

        # Core5 frozen Gated predictions
        gthr=GATED[seed]["normalized_thresholds"]
        bp=1/(1+np.exp(-np.clip(ZB.astype(np.float64),-60,60)))
        core_tasks={}; bpred={}; cacc=[]; cf1=[]
        for k,t in enumerate(BINARY_TASKS):
            d,pr=binary_detail(test_gold_bin[:,k],bp[:,k],gthr["binary"][t])
            core_tasks[t]=d; bpred[t]=pr; cacc.append(d["accuracy"]); cf1.append(d["macro_f1"])
        lp=np.exp(ZL.astype(np.float64)-ZL.max(1,keepdims=True)); lp/=lp.sum(1,keepdims=True)
        ld,lpred=language_detail(test_gold_lang,lp)
        core_tasks["language"]=ld; cacc.append(ld["accuracy"]); cf1.append(ld["macro_f1"])
        c5={"accuracy":float(np.mean(cacc)),"macro_f1":float(np.mean(cf1)),"tasks":core_tasks}

        # Gated Target comparator
        pg=1/(1+np.exp(-np.clip(ZG.astype(np.float64),-60,60)))
        gt=np.asarray([gthr["target"][lab] for lab in TARGET_LABELS],float)
        gated_target,gated_pred=target_metric_bundle(test_gold_target,pg,gt,full=True)

        # FacetGate Target — execute on the same CUDA:0 FP32 specialist path used during
        # validation finalization. This keeps frozen validation thresholds and TEST inference
        # on the same numerical execution surface.
        facet,state=load_facet_state(seed,FACET_INFER_DEVICE)
        probs=[]; sels=[]; views=[]
        with torch.inference_mode():
            for st in range(0,932,128):
                sl=slice(st,min(st+128,932))
                vv=torch.from_numpy(V[sl]).to(FACET_INFER_DEVICE)
                tt=torch.from_numpy(T[sl]).to(FACET_INFER_DEVICE)
                jj=torch.from_numpy(J[sl]).to(FACET_INFER_DEVICE)
                zg=torch.from_numpy(ZG[sl]).to(FACET_INFER_DEVICE)
                zs,ex=facet(vv,tt,jj,zg)
                probs.append(torch.sigmoid(zs).cpu().numpy())
                sels.append(ex["selector"].cpu().numpy())
                views.append(ex["view_weights"].cpu().numpy())
        pf=np.concatenate(probs); sel=np.concatenate(sels); vw=np.concatenate(views)
        ft=np.asarray([FACET[seed]["facetgate_thresholds"]["facetgate_target"][lab] for lab in TARGET_LABELS],float)
        facet_raw,_=target_metric_bundle(test_gold_target,pf,np.full(7,.5),full=True)
        facet_cal,facet_pred=target_metric_bundle(test_gold_target,pf,ft,full=True)

        gated_full6_acc=float((5*c5["accuracy"]+gated_target["exact_set_accuracy"])/6)
        gated_full6_f1=float((5*c5["macro_f1"]+gated_target["macro_f1"])/6)
        full6_acc=float((5*c5["accuracy"]+facet_cal["exact_set_accuracy"])/6)
        full6_f1=float((5*c5["macro_f1"]+facet_cal["macro_f1"])/6)

        lang_rows=language_conditioned(seed,bp,bpred,pf,facet_pred)
        ALL_LANG_ROWS.extend(lang_rows)

        result={
            "schema":"M3Offend_Gemma4_12B_FacetGate_5Seed_FINAL_LOCKED_TEST_SEED_v1_0",
            "lock_id":lock_id,"seed":seed,"model_id":MODEL_ID,"model_revision":MODEL_REVISION,
            "test_rows":932,"test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "facetgate_specialist_test_forwards":932,
            "facetgate_specialist_device":str(FACET_INFER_DEVICE),
            "development_execution_mode":FACET[seed]["run_config"].get("execution_mode"),
            "gated":{
                "selected_step":EXPECTED_GATED_STEP[seed],
                "checkpoint_content_sha256":GATED[seed]["checkpoint_content_sha256"],
                "core5":c5,"target_calibrated":gated_target,
                "full6_accuracy":gated_full6_acc,"full6_macro_f1":gated_full6_f1,
            },
            "facetgate":{
                "selected_step":EXPECTED_FACETGATE_STEP[seed],
                "checkpoint_sha256":FACET[seed]["selected_facetgate_sha256"],
                "validation_threshold_sha256":FACET[seed]["validation_threshold_sha256"],
                "target_raw_0p5":facet_raw,"target_calibrated":facet_cal,
                "full6_accuracy":full6_acc,"full6_macro_f1":full6_f1,
                "target_macro_f1_delta_pp":100*(facet_cal["macro_f1"]-gated_target["macro_f1"]),
                "target_map_delta_pp":100*(facet_cal["map"]-gated_target["map"]),
                "selector_mean":float(sel.mean()),"selector_sd":float(sel.std()),
                "view_weight_mean":dict(zip(["visual","text","product","difference"],vw.mean(0).tolist())),
            },
            "language_conditioned":lang_rows,
            "no_test_tuning":True,
        }

        # Complete prediction ledger
        ledger=pd.DataFrame({
            "official_row_index":expected_test_idx,
            "image_id":test_master.image_id.astype(str).to_numpy(),
            "seed":seed,
            "gold_language":[LANGUAGE_LABELS[i] for i in test_gold_lang],
            "gated_pred_language":[LANGUAGE_LABELS[i] for i in lpred],
        })
        for k,t in enumerate(BINARY_TASKS):
            ledger[f"gold_{t}"]=test_gold_bin[:,k]
            ledger[f"gated_prob_{t}"]=bp[:,k]
            ledger[f"gated_pred_{t}"]=bpred[t]
            ledger[f"gated_threshold_{t}"]=gthr["binary"][t]
        for k,lab in enumerate(LANGUAGE_LABELS):
            ledger[f"gated_prob_language_{lab}"]=lp[:,k]
        for j,lab in enumerate(TARGET_LABELS):
            ledger[f"gold_target_{lab}"]=test_gold_target[:,j]
            ledger[f"gated_target_prob_{lab}"]=pg[:,j]
            ledger[f"gated_target_pred_{lab}"]=gated_pred[:,j]
            ledger[f"gated_target_threshold_{lab}"]=gt[j]
            ledger[f"facetgate_target_prob_{lab}"]=pf[:,j]
            ledger[f"facetgate_target_pred_{lab}"]=facet_pred[:,j]
            ledger[f"facetgate_target_threshold_{lab}"]=ft[j]
            ledger[f"facetgate_selector_{lab}"]=sel[:,j]

        ledger.to_parquet(ledger_path,index=False)
        ledger.to_csv(seed_dir/"locked_test_prediction_ledger.csv",index=False)
        atomic_json(metrics_path,result)
        write_seed_paper_tables(seed_dir,result,lang_rows)
        atomic_json(marker,{
            "schema":"M3Offend_Gemma4_12B_FacetGate_LOCKED_TEST_ONCE_MARKER_v1_0",
            "lock_id":lock_id,"seed":seed,"test_rows":932,"test_model_forwards":932,
            "facetgate_specialist_test_forwards":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "ledger_sha256":sha256_file(ledger_path),"metrics_sha256":sha256_file(metrics_path)
        })
        del M,V,T,J,ZG,ZB,ZL,bp,lp,pg,pf,sel,vw,facet,state
        gc.collect()
        torch.cuda.empty_cache()

    result=json.loads(metrics_path.read_text(encoding="utf-8"))
    PER_SEED_RESULTS.append(result)
    if not reuse:
        # Technical progress package after each completed seed.
        progress_dir=WORK_ROOT/"PROGRESS_COMPLETED_SEEDS"; progress_dir.mkdir(exist_ok=True)
        for s in PUBLICATION_SEEDS:
            src=WORK_ROOT/f"seed_{s}"
            if (src/"LOCKED_TEST_ONCE_MARKER.json").is_file():
                dst=progress_dir/f"seed_{s}"
                if dst.exists(): shutil.rmtree(dst)
                shutil.copytree(src,dst,ignore=shutil.ignore_patterns("locked_test_features","logs"))
        shutil.make_archive(str(WORK_ROOT/"Gemma4_FacetGate_5Seed_LOCKED_TEST_PROGRESS"),"zip",root_dir=progress_dir)
    print("[SEED FINAL LOCKED TEST COMPLETE]",{
        "seed":seed,"Core5_MF1":result["gated"]["core5"]["macro_f1"],
        "Gated_Target_MF1":result["gated"]["target_calibrated"]["macro_f1"],
        "FacetGate_Target_MF1":result["facetgate"]["target_calibrated"]["macro_f1"],
        "FacetGate_Full6_MF1":result["facetgate"]["full6_macro_f1"],
    })

if len(PER_SEED_RESULTS)!=5:
    raise RuntimeError("Five-seed final result count drift")
print("[ALL FIVE GEMMA4 FACETGATE LOCKED TEST SEEDS COMPLETE]",{
    "new_backbone_TEST_forwards_this_session":NEW_TEST_FORWARDS_THIS_SESSION,
    "completed_seed_results":len(PER_SEED_RESULTS),
    "resume_reuse_allowed_only_by_exact_lock_id":True,
})


In [ ]:
# Cell 9 — Complete five-seed paper-data aggregation and final package
# CPU only. No further model forward.
import pandas as pd, numpy as np, math, shutil, json
from scipy.stats import t as student_t

FINAL_DIR=WORK_ROOT/"FINAL_5SEED_RESULTS"
if FINAL_DIR.exists(): shutil.rmtree(FINAL_DIR)
FINAL_DIR.mkdir(parents=True)

def stats(x):
    x=np.asarray(x,float); n=len(x)
    mean=float(x.mean()); sd=float(x.std(ddof=1))
    crit=float(student_t.ppf(.975,n-1)); half=crit*sd/math.sqrt(n)
    return {"n":n,"mean":mean,"sample_sd":sd,"ci95_low":mean-half,"ci95_high":mean+half,"min":float(x.min()),"max":float(x.max())}

def fmt_ms(x):
    z=stats(x); return f"{100*z['mean']:.2f}±{100*z['sample_sd']:.2f}"

# ---------- headline ----------
head=[]
for r in PER_SEED_RESULTS:
    head.append({
        "seed":r["seed"],
        "core5_accuracy":r["gated"]["core5"]["accuracy"],
        "core5_macro_f1":r["gated"]["core5"]["macro_f1"],
        "gated_target_exact_set_accuracy":r["gated"]["target_calibrated"]["exact_set_accuracy"],
        "gated_target_macro_f1":r["gated"]["target_calibrated"]["macro_f1"],
        "gated_target_map":r["gated"]["target_calibrated"]["map"],
        "facetgate_target_exact_set_accuracy":r["facetgate"]["target_calibrated"]["exact_set_accuracy"],
        "facetgate_target_macro_f1":r["facetgate"]["target_calibrated"]["macro_f1"],
        "facetgate_target_map":r["facetgate"]["target_calibrated"]["map"],
        "facetgate_target_micro_f1":r["facetgate"]["target_calibrated"]["micro_f1"],
        "facetgate_target_samples_f1":r["facetgate"]["target_calibrated"]["samples_f1"],
        "facetgate_target_hamming_loss":r["facetgate"]["target_calibrated"]["hamming_loss"],
        "facetgate_rare_target_macro_f1":r["facetgate"]["target_calibrated"]["rare_target_macro_f1"],
        "target_macro_f1_delta_pp":r["facetgate"]["target_macro_f1_delta_pp"],
        "target_map_delta_pp":r["facetgate"]["target_map_delta_pp"],
        "gated_full6_accuracy":r["gated"]["full6_accuracy"],
        "gated_full6_macro_f1":r["gated"]["full6_macro_f1"],
        "facetgate_full6_accuracy":r["facetgate"]["full6_accuracy"],
        "facetgate_full6_macro_f1":r["facetgate"]["full6_macro_f1"],
    })
per_seed=pd.DataFrame(head).sort_values("seed").reset_index(drop=True)
per_seed.to_csv(FINAL_DIR/"gemma4_facetgate_locked_test_per_seed.csv",index=False)

summary_rows=[]
for col in per_seed.columns:
    if col=="seed": continue
    summary_rows.append({"metric":col,**stats(per_seed[col])})
summary=pd.DataFrame(summary_rows)
summary.to_csv(FINAL_DIR/"gemma4_facetgate_locked_test_mean_sd_ci95.csv",index=False)

# ---------- taskwise, binary classwise, language classwise ----------
task_rows=[]; binary_rows=[]; lang_class_rows=[]; target_rows=[]; langcond_rows=[]
for r in PER_SEED_RESULTS:
    seed=r["seed"]; c=r["gated"]["core5"]
    for task in [*BINARY_TASKS,"language"]:
        d=c["tasks"][task]
        task_rows.append({"seed":seed,"task":task,"accuracy":d["accuracy"],"macro_f1":d["macro_f1"]})
    for task in BINARY_TASKS:
        d=c["tasks"][task]
        for cls in ["no","yes"]:
            binary_rows.append({
                "seed":seed,"task":task,"class":cls.title(),"support":d[f"support_{cls}"],
                "precision":d[f"precision_{cls}"],"recall":d[f"recall_{cls}"],"f1":d[f"f1_{cls}"]
            })
    ld=c["tasks"]["language"]["per_class"]
    for lab in LANGUAGE_LABELS:
        lang_class_rows.append({"seed":seed,"language":lab,**ld[lab]})
    gd=r["gated"]["target_calibrated"]["per_label"]; fd=r["facetgate"]["target_calibrated"]["per_label"]
    for lab in TARGET_LABELS:
        g=gd[lab]; f=fd[lab]
        target_rows.append({
            "seed":seed,"label":lab,"support":f["support"],
            "gated_precision":g["precision"],"gated_recall":g["recall"],"gated_f1":g["f1"],"gated_ap":g["ap"],"gated_threshold":g["threshold"],
            "facetgate_precision":f["precision"],"facetgate_recall":f["recall"],"facetgate_f1":f["f1"],"facetgate_ap":f["ap"],"facetgate_threshold":f["threshold"],
            "delta_f1_pp":100*(f["f1"]-g["f1"]),"delta_ap_pp":100*(f["ap"]-g["ap"]),
        })
    langcond_rows += r["language_conditioned"]

task_df=pd.DataFrame(task_rows); binary_df=pd.DataFrame(binary_rows)
langclass_df=pd.DataFrame(lang_class_rows); target_df=pd.DataFrame(target_rows)
langcond_df=pd.DataFrame(langcond_rows)
task_df.to_csv(FINAL_DIR/"all_seed_core5_taskwise.csv",index=False)
binary_df.to_csv(FINAL_DIR/"all_seed_binary_classwise.csv",index=False)
langclass_df.to_csv(FINAL_DIR/"all_seed_language_classwise.csv",index=False)
target_df.to_csv(FINAL_DIR/"all_seed_target_labelwise.csv",index=False)
langcond_df.to_csv(FINAL_DIR/"all_seed_language_conditioned.csv",index=False)

def agg_long(df,group_cols,value_cols):
    out=[]
    for key,g in df.groupby(group_cols,sort=False):
        if not isinstance(key,tuple): key=(key,)
        base=dict(zip(group_cols,key))
        for col in value_cols:
            z=stats(g[col].astype(float).to_numpy())
            out.append({**base,"metric":col,**z})
    return pd.DataFrame(out)

agg_long(task_df,["task"],["accuracy","macro_f1"]).to_csv(FINAL_DIR/"paper_core5_taskwise_mean_sd_ci95.csv",index=False)
agg_long(binary_df,["task","class"],["precision","recall","f1"]).to_csv(FINAL_DIR/"paper_binary_classwise_mean_sd_ci95.csv",index=False)
agg_long(langclass_df,["language"],["precision","recall","f1"]).to_csv(FINAL_DIR/"paper_language_classwise_mean_sd_ci95.csv",index=False)
agg_long(target_df,["label"],[
    "gated_precision","gated_recall","gated_f1","gated_ap",
    "facetgate_precision","facetgate_recall","facetgate_f1","facetgate_ap",
    "delta_f1_pp","delta_ap_pp"
]).to_csv(FINAL_DIR/"paper_target_labelwise_mean_sd_ci95.csv",index=False)

# ---------- language-conditioned tables exactly in paper/image style ----------
langagg=agg_long(langcond_df,["language"],[
    "offensive_accuracy","offensive_macro_f1","vulgar_accuracy","vulgar_macro_f1",
    "abuse_accuracy","abuse_macro_f1","sarcastic_accuracy","sarcastic_macro_f1",
    "four_task_accuracy","four_task_macro_f1","target_exact_set_accuracy","target_macro_f1",
    "target_map","five_task_accuracy","five_task_macro_f1"
])
langagg.to_csv(FINAL_DIR/"paper_language_conditioned_all_metrics_mean_sd_ci95.csv",index=False)

paper_mf1=[]
paper_acc=[]
for lang in LANGUAGE_LABELS:
    g=langcond_df[langcond_df.language.eq(lang)]
    abbrev={"Bangla":"BN","English":"EN","Banglish":"Bgl"}[lang]
    paper_mf1.append({
        "Model":"Gemma4","Lang.":abbrev,
        "Off.":fmt_ms(g.offensive_macro_f1),"Vul.":fmt_ms(g.vulgar_macro_f1),
        "Abuse":fmt_ms(g.abuse_macro_f1),"Sarc.":fmt_ms(g.sarcastic_macro_f1),
        "4-task":fmt_ms(g.four_task_macro_f1),"Target":fmt_ms(g.target_macro_f1),
        "5-task":fmt_ms(g.five_task_macro_f1),
    })
    paper_acc.append({
        "Model":"Gemma4","Lang.":abbrev,
        "Off.":fmt_ms(g.offensive_accuracy),"Vul.":fmt_ms(g.vulgar_accuracy),
        "Abuse":fmt_ms(g.abuse_accuracy),"Sarc.":fmt_ms(g.sarcastic_accuracy),
        "4-task":fmt_ms(g.four_task_accuracy),"Target":fmt_ms(g.target_exact_set_accuracy),
        "5-task":fmt_ms(g.five_task_accuracy),
    })
pd.DataFrame(paper_mf1).to_csv(FINAL_DIR/"PAPER_TABLE_language_conditioned_macro_f1_percent.csv",index=False)
pd.DataFrame(paper_acc).to_csv(FINAL_DIR/"PAPER_TABLE_language_conditioned_accuracy_percent.csv",index=False)

# ---------- headline summary ----------
paper_head=[]
for metric in [
    "core5_accuracy","core5_macro_f1","gated_target_macro_f1","facetgate_target_exact_set_accuracy",
    "facetgate_target_macro_f1","facetgate_target_map","target_macro_f1_delta_pp",
    "facetgate_full6_accuracy","facetgate_full6_macro_f1"
]:
    z=stats(per_seed[metric])
    scale=1.0 if metric.endswith("_pp") else 100.0
    paper_head.append({
        "metric":metric,
        "mean":scale*z["mean"],"sample_sd":scale*z["sample_sd"],
        "ci95_low":scale*z["ci95_low"],"ci95_high":scale*z["ci95_high"],
        "paper_mean_sd":f"{scale*z['mean']:.2f} ± {scale*z['sample_sd']:.2f}"
    })
pd.DataFrame(paper_head).to_csv(FINAL_DIR/"PAPER_TABLE_headline_mean_sd_ci95.csv",index=False)

# Copy all seed ledgers and paper outputs.
for seed in PUBLICATION_SEEDS:
    src=WORK_ROOT/f"seed_{seed}"; dst=FINAL_DIR/f"seed_{seed}"
    dst.mkdir()
    for name in [
        "locked_test_metrics.json","locked_test_prediction_ledger.parquet","locked_test_prediction_ledger.csv",
        "LOCKED_TEST_ONCE_MARKER.json","locked_test_worker_config.json",
        "core5_taskwise.csv","binary_classwise.csv","language_classwise.csv","target_labelwise.csv","language_conditioned.csv"
    ]:
        p=src/name
        if p.is_file(): shutil.copy2(p,dst/name)

payload={
    "schema":"M3Offend_Gemma4_12B_FacetGate_5Seed_FINAL_LOCKED_TEST_SUMMARY_v1_0",
    "build_id":LOCKED_TEST_BUILD_ID,
    "model_id":MODEL_ID,"model_revision":MODEL_REVISION,
    "facetgate_protocol_sha256":FACETGATE_PROTOCOL_SHA256,
    "gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
    "test_rows_per_seed":932,"test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "seeds":PUBLICATION_SEEDS,
    "statistics":"arithmetic mean + sample SD (ddof=1) + two-sided 95% Student-t CI across five predefined seeds",
    "core5_policy":"frozen same-seed Gated branch and validation thresholds",
    "target_policy":"validation-selected FacetGate checkpoint + seven validation-selected Target thresholds",
    "language_conditioned_definition":"gold-language subgroup; 4-task=Offensive+Vulgar+Abuse+Sarcastic; 5-task=4-task+Target; Language classification excluded",
    "no_test_tuning":True,"no_selective_rerun":True,
    "validation_replay_device":str(REPLAY_DEVICE),
    "validation_map_replay_tolerance":MAP_REPLAY_ABS_TOL,
    "per_seed":head,
    "summary":{r["metric"]:{k:r[k] for k in ["n","mean","sample_sd","ci95_low","ci95_high","min","max"]} for r in summary_rows},
}
atomic_json(FINAL_DIR/"gemma4_facetgate_5seed_final_locked_test_summary.json",payload)

manifest={"schema":"M3Offend_Gemma4_12B_FacetGate_FINAL_PACKAGE_MANIFEST_v1_0","files":{}}
for p in sorted(FINAL_DIR.rglob("*")):
    if p.is_file() and p.name!="FINAL_PACKAGE_MANIFEST.json":
        manifest["files"][p.relative_to(FINAL_DIR).as_posix()]=sha256_file(p)
atomic_json(FINAL_DIR/"FINAL_PACKAGE_MANIFEST.json",manifest)

zip_base=WORK_ROOT/"M3Offend_9318_Gemma4_12B_FacetGate_5Seed_FINAL_LOCKED_TEST_ALL_PAPER_DATA_v1_0"
zip_path=Path(shutil.make_archive(str(zip_base),"zip",root_dir=FINAL_DIR))

print("\n=== GEMMA4 FACETGATE FIVE-SEED FINAL LOCKED TEST — HEADLINE ===")
display(pd.DataFrame(paper_head))
print("\n=== LANGUAGE-CONDITIONED MACRO-F1 PAPER TABLE ===")
display(pd.DataFrame(paper_mf1))
print("\n=== LANGUAGE-CONDITIONED ACCURACY PAPER TABLE ===")
display(pd.DataFrame(paper_acc))
print("\nFinal ZIP:",zip_path)
print("Final ZIP SHA256:",sha256_file(zip_path))
